# Mem0 · A Running Coach That Remembers

A coaching app: runners check in once a week with how training went and a question, and the coach replies. A good human coach remembers each runner — their race, their goal, which days they can run, what hurts — and a coach that forgets makes every check-in start from nothing.

What makes this hard is not storing facts. It is that **the facts move**. Over six weeks, Maya's situation changes five times:

| When | What changes |
|---|---|
| 15 Sep | her race is put back two weeks, from **18 October to 1 November**; she switches from **evening runs to mornings**; her knee is sore after hills |
| 22 Sep | a physio **clears the knee**; a new job cuts her from **four running days to three** (Tue, Fri, Sun) |
| 29 Sep | a 10k personal best moves her goal from **1:50 to 1:45** |

So at her last check-in, on 13 October, she asks the coach to remind her of everything, and every answer has an old version that would now be wrong. A second runner, **Tom**, training for a marathon, checks in alongside her. Nothing of Maya's may ever reach Tom's coaching.

**Mem0** is a memory layer for exactly this: after each conversation, its own model pulls out the facts worth keeping, and before the next reply, a search brings back the ones that bear on it. The coach is built twice, with the same coach model (gpt-oss-120b on Groq), the same persona and the same nine scripted check-ins: **with Mem0** (`coach_mem0.py`), and **from this repo's parts** (`coach_scratch.py`), using chapter 4's design for memory.

| File | What it is |
|---|---|
| `runners.py` | the two runners, the nine check-ins, and what a reply that needs memory should and should not say |
| `coach_mem0.py` | the coach with Mem0 |
| `coach_scratch.py` | the coach from this repo's parts |
| `coach_stubs.py` | stand-ins for every model and embedding call, so both coaches run offline through real Mem0, Qdrant, SQLite |
| `coach_checks.py` | what building it found out about Mem0, each reproducible offline, and the views used below |
| `test_coach.py` | offline tests for all of it, and for chapter 4's new `QdrantJournal` |

---
# Part 1 · Build

## The coach with Mem0

Mem0's loop, as its documentation gives it, once per check-in:

1. **search** the runner's memories, using their message as the query
2. put what comes back into the coach's prompt, and **reply**
3. **add** the exchange — Mem0's own model reads it and decides what to keep

The coach never decides what to remember; Mem0 does it after every check-in. One runner is told from another by `user_id`, which every search must name in `filters`.

```python
# coach_mem0.py
def recall(m, session):
    """Mem0's search for this runner, with the check-in's own words as the query -- its defaults otherwise.

    `explain=True` changes nothing about what comes back; it keeps the three parts of each score. The score itself
    is (meaning + keywords + entities) divided by however many of those three fired for this query, so it is not a
    similarity, and two searches' scores are not on one scale. Mem0's 0.1 threshold is applied to the meaning part
    alone, before the other two are added."""
    found = m.search(session["message"], filters={"user_id": session["runner"]}, explain=True)["results"]
    return [{"memory": r["memory"], "score": round(r["score"], 3), "session": (r.get("metadata") or {}).get("session"),
             "parts": {k: round(r["score_details"][v], 3) for k, v in
                       (("meaning", "semantic_score"), ("keywords", "bm25_score"), ("entities", "entity_boost"))}}
            for r in found]

def system(session, recalled):
    """The coach's persona, and what Mem0 brought back -- each memory with the check-in it came from."""
    lines = [f"- ({r['session']}) {r['memory']}" for r in recalled] or ["- nothing yet"]
    return (R.COACH.format(date=session["date"], name=R.RUNNERS[session["runner"]]) +
            f"\n\nWhat you remember about {R.RUNNERS[session['runner']]} from earlier check-ins:\n" + "\n".join(lines))

def remember(m, session, reply_text, instructions=""):
    """Mem0's `add` over the check-in, dated through custom instructions (plus any more). Returns what it stored."""
    exchange = [{"role": "user", "content": session["message"]}, {"role": "assistant", "content": reply_text}]
    calls = len(m.llm.calls)
    added = m.add(exchange, user_id=session["runner"], metadata={"session": session["date"]},
                  prompt=DATED.format(date=session["date"]) + instructions)["results"]
    return {"memories": [a["memory"] for a in added], "events": sorted({a["event"] for a in added}),
            "memory model": m.llm.calls[calls:]}
```

`explain=True` changes nothing about what comes back. It keeps the three parts of each score, which matter later: Mem0 ranks memories by **meaning** (embeddings), **keywords** (BM25) and **entities** (names spaCy finds, linked to the memories that mention them).

### What `add` does, in Mem0 2.2.1

One call to Mem0's model, with a long system prompt whose first instruction is *"Your sole operation is ADD"*. Mem0 used to have a second step that decided, for each new fact, whether to add it, update an old memory or delete one. In 2.x that step is gone, although `add`'s docstring still describes it. A fact that changes is written as **a new memory saying what changed** ("the race was moved from 18 October to 1 November"), and the old memory stays. The only duplicates removed are memories with exactly the same text.

So when Maya's race moves, Mem0 holds both dates, and a search for her race can bring back both. Working out which is current is left to the coach's model, at reply time. That is why each memory in the coach's prompt carries the date of the check-in it came from.

### Setting it up, and why each choice

- **Mem0's model is Gemini flash-lite, not Groq.** Mem0's extraction prompt is about 7,600 tokens on its own, and Groq's free tier refuses any request over 8,000. Add the conversation and room for the reply, and every `add` is refused. The coach itself stays on Groq, the same as the scratch build. The comparison is between two ways of keeping memory, with one coach.
- **Qdrant, fastembed and spaCy.** Mem0's search needs all three for its three scores. Qdrant is its default store and the only one of its local stores with keyword search. Mem0 uses that search only if `fastembed` is installed, and entities only if spaCy is. With Chroma, or without those packages, search falls back to meaning alone, with nothing more than a warning line.
- **The date of each check-in** goes in through `prompt` (Mem0's custom instructions) and `metadata`. The `timestamp` argument raises an error in the open-source build.
- **Telemetry off and a project folder for Mem0's files.** Both are on by default and both are shared machine-wide; Part 2 shows the defaults.

```python
# coach_mem0.py
def config(root=ROOT, key=None):
    key = key or os.environ.get("GEMINI_API_KEY")
    return {
        "vector_store": {"provider": "qdrant",
                         "config": {"collection_name": "coach", "path": str(root / "qdrant"),
                                    "embedding_model_dims": 768, "on_disk": True}},
        # the key is passed, not left to Mem0: its Gemini classes fall back to GOOGLE_API_KEY, which on this machine
        # is a different key from the one in .env
        "llm": {"provider": "gemini", "config": {"model": MEMORY_MODEL, "api_key": key, "max_tokens": MEMORY_TOKENS}},
        "embedder": {"provider": "gemini", "config": {"model": EMBED_MODEL, "api_key": key, "embedding_dims": 768}},
        "history_db_path": str(root / "history.db"),
    }

def memory(root=ROOT, llm=None, embedder=None, key=None):
    """Mem0's `Memory` on this project's store, its model counted and its embeddings kept."""
    from mem0 import Memory
    root.mkdir(parents=True, exist_ok=True)
    m = Memory.from_config(config(root, key))
    m.llm = Counted(llm or m.llm)
    m.embedding_model = Cached(embedder or m.embedding_model, root / "embeddings.json")
    return m
```

`Cached` keeps each embedding on disk. Gemini returns the same numbers for the same text every time, so it changes nothing Mem0 sees, and the free tier's 1,000 embedded texts a day go on new text only. `Counted` counts Mem0's model calls, and waits out Gemini's "busy" and per-minute refusals instead of failing an `add`.

`run` goes through the check-ins in order and saves each step as it finishes. A run stopped by a quota carries on where it stopped, and never asks the coach again for a reply it already has.

```python
# coach_mem0.py
def run(m, sessions=R.SESSIONS, path=ANSWERS, log=print, instructions=""):
    """Every check-in in order: recall, reply, remember -- each step saved as it finishes, so a run stopped by a quota
    carries on where it stopped and never asks the coach again for a reply it already has."""
    done = _load(path)
    for session in sessions:
        record = done.setdefault(session["id"], {"runner": session["runner"], "date": session["date"]})
        if "reply" not in record:
            record["recalled"] = recall(m, session)
            record["reply"], record["usage"] = reply(system(session, record["recalled"]), session["message"])
            _save(path, done)
        if "added" not in record:
            try:
                record["added"] = remember(m, session, record["reply"], instructions)
            except Exception as error:      # noqa: BLE001 -- Mem0 wraps the model's error in its own LLMError
                quota = isinstance(error.__cause__, MemoryQuotaSpent) or isinstance(error, MemoryQuotaSpent)
                log(f"  {session['id']}: add failed -- {'daily quota' if quota else type(error).__name__}: "
                    f"{str(error)[:200]}")
                return done, False
            _save(path, done)
        log(f"  {session['id']}: recalled {len(record['recalled'])}, stored {len(record['added']['memories'])}")
    return done, True
```

## The same coach, from scratch: chapter 4's design

Mem0 keeps every memory the same way: a sentence, embedded, found again by search. Chapter 4 decides where a memory goes by **what will be asked of it later**, and a coach asks three different things:

| What the coach asks | The kind of memory | Where it goes |
|---|---|---|
| "when is her race, what is her goal, which days does she run" | facts with names | chapter 4's `KeyValue` — a new value **replaces** the old one, so a race that moves leaves one date, not two |
| "how have her long runs built up" | events | a run log in the shape of chapter 4's `Records` (SQLite): filtered, ordered, **always appended** |
| anything else worth knowing next time | notes nobody could design a field for | chapter 4's `QdrantJournal`, found again by meaning through chapter 4's `Recall` and its floor |

The run log being append-only matters. Two long runs a week apart look like duplicates to a similarity check, and they are two events: chapter 4's types-of-memory notebook showed an add/update/ignore check destroying exactly this kind of history.

`QdrantJournal` is new in chapter 4's `stores.py`. It is chapter 4's `Journal` on one engine instead of two: Qdrant keeps the keyword index and the vectors on one record and fuses the two searches itself. It has the same methods, so `Recall` runs on it unchanged. The offline tests check the two journals give the same answers. They differ in two ways: Qdrant's BM25 reduces words to their stems ("batches" finds "batch"), and its own fusion constant is 2 instead of chapter 4's 60, so the 60 is passed in.

The coach uses memory the way chapter 4's last notebook settled on: **recall by injection, write by tool**.

- **Reading.** `Recollect`, a loop component before Think like chapter 4's `Retrieve`, shows the coach the whole profile, the run log, and the notes that bear on the message, at every check-in.
- **Writing.** One `remember` tool. The coach decides what is worth keeping; the code decides where each piece goes.

```python
# coach_scratch.py
PROFILE = {"race": "the race they are training for, and its date",
           "goal": "their goal time",
           "days": "the days of the week they run",
           "time_of_day": "when in the day they run",
           "injuries": "anything that hurts, and how it stands now",
           "food": "their diet, and what they eat and drink on runs"}
WRITE = """
Before you reply, call `remember` once with what in the message is worth keeping for later check-ins:
- profile: a field whose value has changed or is new. Write the whole current value -- it replaces the old one.
- runs: every run the runner reports, with its date worked out from today's date.
- notes: anything else worth knowing at a later check-in that no profile field holds.
If nothing in the message is new, reply without calling it."""
```

```python
# coach_scratch.py
def keep(self, today, profile=(), runs=(), notes=()):
    """The routing: each kind of memory to its store, the way that kind is written."""
    said = []
    for change in profile:
        old = self.profile.read(change.field) or {}
        self.profile.write(change.field, {"value": change.value, "since": today,
                                          "was": old.get("value"), "was_since": old.get("since")})
        said.append(f"{change.field} = {change.value}")
    logged = sum(self.runs.write(run) for run in runs)
    if runs:
        said.append(f"{logged} run(s) logged" + (f", {len(runs) - logged} already there" if logged < len(runs) else ""))
    known = {row["text"] for row in self.notes.recent(limit=1000)}
    fresh = [n for n in dict.fromkeys(notes) if n not in known]
    if fresh:
        self.notes.write([(today, self.runner, "coach", n) for n in fresh], now=today)
        said.append(f"{len(fresh)} note(s) kept")
    return "Kept: " + "; ".join(said) if said else "Nothing new kept."

def render(self, today, message, floor=0.6, share=None, cap=None, search="meaning"):
    """What the coach is shown before it replies: all of the profile, the runs, and the notes that bear on this.

    search="meaning"    what rounds 1-7 ran: the top 3 above the floor, by meaning alone (or with `share` and
                        `cap`, that share of all notes, at most `cap`)
    search="questions"  chapter 4's hybrid `Recall`, one search per question in the message, merged; and every
                        habit note shown as well, searched or not -- a runner has few, and they bear on every
                        check-in"""
    from recall import Recall, questions
    name = R.RUNNERS[self.runner]
    profile, runs = self.lines()
    habits = []
    if not len(self.notes):
        notes = []
    elif search == "questions":
        habit_rows = self.notes.recent(source="habit", limit=1000)
        habits = [f"- {row['at']} {row['text']}" for row in habit_rows]
        picked = Recall(self.notes, today=today, floor=floor).many(questions(message), k=3, share=share, cap=cap)
        shown = {row["id"] for row in habit_rows}
        notes = [f"- {at} {text}" for _, eid, at, text in picked if eid not in shown]
    else:
        recall = Recall(self.notes, today=today, floor=floor, hybrid=False)
        notes = [f"- {at} {text}" for _, _, at, text in recall(message, k=3, share=share, cap=cap)]
    if not (profile or runs or notes or habits):
        return None
    parts = [f"What you know about {name} from earlier check-ins."]
    parts += ["Profile:", *profile] if profile else ["Profile: nothing yet"]
    parts += ["Runs, oldest first:", *runs] if runs else ["Runs: none logged yet"]
    if habits:
        parts += ["Habits:", *habits]
    if notes:
        parts += ["Notes that bear on this message:", *notes]
    return "\n".join(parts)
```

```python
# coach_scratch.py
def remember_tool(memory, today, log=None):
    """`remember`, closed over one runner's stores and today's date -- a parameter the model has no way to know."""
    from tools import tool

    @tool
    def remember(profile: Annotated[Optional[list[Change]], "profile fields that changed or are new"] = None,
                 runs: Annotated[Optional[list[Run]], "every run reported in the message"] = None,
                 notes: Annotated[Optional[list[str]], "anything else worth knowing at a later check-in"] = None
                 ) -> str:
        """Keep what is worth knowing at later check-ins with this runner."""
        to_model = lambda items, cls: [cls.model_validate(i) if isinstance(i, dict) else i for i in items or []]
        said = memory.keep(today, to_model(profile, Change), to_model(runs, Run), notes or [])
        if log is not None:
            log.append({"profile": profile or [], "runs": runs or [], "notes": notes or [], "said": said})
        return said

    return remember

def coach(memory, today, log=None, model=R.COACH_MODEL, write="tool"):
    """The loop: Recollect once, then Think -> Act -> Observe until the coach replies. With write="step" or
    "checked" the coach has no tool and only replies; the writing is done afterwards."""
    from act import Act
    from observe import Observe
    from runtime import Runtime
    from supervisor import Think, verbatim
    recollect = Recollect(memory, today)
    if write in ("step", "checked", "card", "card5"):
        # Act stays, with nothing to call: Think's answer is a "respond" decision, and Act is what turns it into
        # the reply that ends the run
        return Runtime(loop=[recollect, Think(model=model, tools=[], allow_fork=False, explain=False,
                                              max_tokens=R.REPLY_TOKENS), Act()],
                       repeat_from=1, max_iterations=7), recollect
    remember = remember_tool(memory, today, log)
    runtime = Runtime(loop=[recollect,
                            Think(model=model, tools=[remember.schema], allow_fork=False, explain=False,
                                  max_tokens=R.REPLY_TOKENS),
                            Act(functions={"remember": remember.function}, schemas={"remember": remember.args}),
                            Observe(extractors={"remember": verbatim})],
                      repeat_from=1, max_iterations=13)       # four turns: more than one write and a reply need
    return runtime, recollect
```

Each runner has a folder of their own, so two runners' memories are never filtered apart: they are never together. The loop is chapter 2's `Runtime`, with chapter 6's `Think` (an output ceiling that can be set) and `verbatim`, so what the tool returns reaches the coach unchanged.

## How the replies are checked

Five check-ins need memory to answer well: Maya's on 8 and 22 September, 29 September and 13 October, and Tom's on 30 September. For each, `runners.CHECKS` lists the facts the reply needs and the old facts that would be wrong now. For Tom, it also lists Maya's facts, which must not appear at all.

Most of these test facts with names that change, which is what the scratch build's profile is designed for. Tom's last check-in tests the other side. He also asks what to eat on race morning, and the answer depends on something he said in passing two weeks earlier, in a message about something else: he gets stomach cramps if he eats less than two hours before a run. Nobody would design a profile field for that. Catching it is what Mem0's extraction is for, while the scratch coach has to decide to write it down.

These lists only point at where to look. A reply can mention 18 October correctly ("moved from 18 October to 1 November"), so every reply is **read in full**, and the reading is what the findings report.

---
# Part 2 · Run and show

In [1]:
import logging
import warnings

warnings.filterwarnings("ignore")
logging.getLogger("google_genai").setLevel(logging.ERROR)    # "Both GOOGLE_API_KEY and GEMINI_API_KEY are set": the key is passed explicitly

import coach_checks as K
import coach_mem0 as M
import coach_scratch as S
import coach_stubs as T
import llm  # chapter 1's client; loads .env
import runners as R

## 1 · What you have to know about Mem0

Each of these is reproduced offline, from Mem0 itself: a real `Memory` on a real Qdrant store, with stand-ins only for the model and the embedder.

**It only ever adds.** Maya's race moves between her first and third check-ins. Afterwards Mem0 holds both versions, and its history has never recorded anything but `ADD`:

In [2]:
K.show(K.adds_only())

{
 "race memories held": [
  "Maya is training for the Riverside Half Marathon on Sunday 18 October 2026 and wants to finish under 1:50",
  "The Riverside Half Marathon was moved from 18 October to Sunday 1 November 2026 because of roadworks"
 ],
 "events in history": [
  "ADD"
 ],
 "add's docstring still says": "infer (bool, optional): If True (default), an LLM is used to extract key facts from 'messages' and decide whether to add, update, or delete related memories.",
 "the prompt says": "Your sole operation is ADD: identify every piece of memorable information and produce self-contained, contextually rich factual statements"
}


**Dates.** The open-source build refuses `timestamp`, and the model is told that today is the date of the conversation. The check-in's real date reaches it only as a custom instruction, which the prompt ranks highest:

In [3]:
K.show(K.dates())

{
 "add(timestamp=...)": "The timestamp parameter is not supported by the OSS Memory SDK.",
 "what the model is told": [
  "## Observation Date",
  "2026-10-10",
  "## Current Date",
  "2026-10-10",
  "## Custom Instructions",
  "This conversation took place on 2026-09-01. Resolve every relative date in it ('last Sunday', 'on Saturday') against 2026-09-01 -- not against the Observation Date or the Current Date.",
  "# Output:"
 ]
}


**Too big for Groq's free tier.** The extraction prompt alone, then the largest request one of these check-ins makes, then that plus the 2,000 output tokens Mem0 asks for by default:

In [4]:
K.show(K.prompt_size())

{
 "extraction prompt alone": 7671,
 "largest add request seen here": 8293,
 "plus Mem0's own output allowance (its default max_tokens)": 10293,
 "Groq's free-tier limit per request": 8000
}


**Three scores, two of them optional.** One search result's parts on Qdrant with fastembed and spaCy installed; Chroma's keyword search (Mem0 prints a warning when it is built on Chroma); and the entities spaCy finds in a question.

The **0.1 threshold applies to the meaning part alone**, before keywords and entities are added. The final score is divided by however many parts fired for that query, so it is not a similarity, and two searches' scores are not on one scale.

In [5]:
K.show(K.search())

The 'chroma' vector store does not support keyword search. Hybrid (BM25) scoring will be disabled and search will use semantic similarity only. To enable hybrid search, switch to a store with keyword_search support (e.g. qdrant, elasticsearch, pgvector).


{
 "Qdrant, one result's parts": {
  "semantic_score": 0.274,
  "bm25_score": 0.336,
  "entity_boost": 0.483,
  "raw_score": 1.092,
  "max_possible_score": 2.5,
  "final_score": 0.437,
  "threshold": 0.1
 },
 "Chroma's keyword search": null,
 "spaCy's entities in a question": [
  [
   "PROPER",
   "Maya"
  ],
  [
   "PROPER",
   "Riverside Half Marathon"
  ],
  [
   "PROPER",
   "November"
  ]
 ]
}


**Scoping.** One runner is told from another only by the filter. `get_all` stops at 20 without saying so:

In [6]:
K.show(K.scoping())

{
 "search(user_id=...)": "Top-level entity parameters frozenset({'user_id'}) are not supported in search(). Use filters={'user_id': '...'} instead",
 "memories stored for one user": 25,
 "get_all with its default top_k": 20,
 "Tom's search, in a store holding only Maya": []
}


**Defaults worth changing.** Telemetry goes to PostHog unless `MEM0_TELEMETRY` is set to false, and the history database lands in one machine-wide folder unless `MEM0_DIR` or `history_db_path` says otherwise. Mem0's Gemini embedder ignores whether a text is being stored or searched for. Chapter 4 found that this distinction matters: Gemini embeds the same sentence differently for each.

In [7]:
K.show(K.defaults())

{
 "telemetry, unless MEM0_TELEMETRY says otherwise": "on",
 "telemetry here": "off",
 "where the history database goes by default": [
  "mem0_dir = os.environ.get(\"MEM0_DIR\") or os.path.join(home_dir, \".mem0\")",
  "default=os.path.join(mem0_dir, \"history.db\"),"
 ],
 "Gemini embedder uses memory_action": false
}


**Links asked for and thrown away.** The prompt asks the model to link each new memory to the old ones it relates to, which costs output tokens on every `add`. Nothing in `add` reads the links back:

In [8]:
K.show(K.links())

{
 "the prompt asks for linked_memory_ids": true,
 "add reads them from the model's reply": false
}


## 2 · Both coaches offline, and what the live run will cost

Both builds over all nine check-ins with every model call stood in. These are the requests each coach would send to Groq (prompt sizes only, since the stand-ins' replies are one line), the number of calls Mem0 makes to its own model, and the texts it embeds:

In [9]:
K.show(K.estimate())

{
 "scratch coach": {
  "requests": 16,
  "largest": 1450,
  "sent in all": 19911
 },
 "Mem0 coach": {
  "requests": 9,
  "largest": 554,
  "sent in all": 2986
 },
 "Mem0's model": {
  "requests": 9
 },
 "texts Mem0 embeds": 68
}


## 3 · Live: the coach with Mem0

Nine check-ins in order. Run the cell again after a quota stop: it carries on from the step that stopped.

In [2]:
mem0 = M.memory()
with_mem0, finished = M.run(mem0)
print("\nall nine done" if finished else "\nstopped -- run this cell again to carry on")
print("new texts embedded this run:", mem0.embedding_model.embedded)

  maya-1: recalled 0, stored 7


  tom-1: recalled 0, stored 5


  maya-2: recalled 7, stored 5


  maya-3: recalled 12, stored 6


  tom-2: recalled 5, stored 6


  maya-4: recalled 18, stored 4


  maya-5: recalled 20, stored 4


  tom-3: recalled 11, stored 2


  maya-6: recalled 20, stored 0

all nine done
new texts embedded this run: 155


## 4 · Live: the coach from scratch

In [2]:
from_scratch, finished = S.run()
print("\nall nine done" if finished else "\nstopped -- run this cell again to carry on")

  maya-1: kept 1 time(s), 2 calls, 3,212 tokens


  tom-1: kept 1 time(s), 2 calls, 2,326 tokens


  maya-2: kept 1 time(s), 2 calls, 2,576 tokens


  maya-3: kept 1 time(s), 2 calls, 3,137 tokens


  tom-2: kept 1 time(s), 2 calls, 2,407 tokens


  maya-4: kept 1 time(s), 3 calls, 4,299 tokens


  maya-5: kept 1 time(s), 2 calls, 3,760 tokens


  tom-3: kept 0 time(s), 1 calls, 1,774 tokens


  maya-6: kept 0 time(s), 1 calls, 1,304 tokens

all nine done


## 5 · The replies that need memory, side by side

Under each reply, what the check points at in it: needed facts mentioned or missing, and old or other-runner facts mentioned. These are places to look, and the reading comes in the findings.

In [3]:
K.side_by_side("maya-2", with_mem0, from_scratch)

maya-2, 2026-09-08 -- Maya:
  Sunday's long run was 16 km in 1:38 -- felt good, and the knee was fine. Tuesday and Thursday were easy 6 km runs.
  Should I add a tempo run, and if so, on which day?

---- the coach with memory Mem0 ----
Great job on the 16 km long run—1:38 is solid and it’s reassuring that the knee stayed pain‑free!  

**Tempo addition**  
- **When?** Keep the tempo on **Tuesday** as originally planned. It’s the best day to slot a quality effort because you still have Thursday for a lighter speed/hill session and Saturday to recover before the next long run.  
- **What to run?**  
  1. **Warm‑up:** 1 km easy (10 spm + 10‑15 sec per stride)  
  2. **Tempo block:** 5 km at “comfortably hard” – aim for ~4:45‑5:00 min/km (roughly 10‑15 sec slower than your 5 km race pace). Keep cadence in the 85‑90 spm range to protect the knee.  
  3. **Cool‑down:** 1 km easy + 5 min of light stretching/foam‑roll.  

**Thursday** – keep it an easy 6 km or turn it into a short hill/stride w

In [4]:
K.side_by_side("maya-4", with_mem0, from_scratch)

maya-4, 2026-09-22 -- Maya:
  Good news: I saw a physio and my knee is fine -- it was a tight IT band, and I've been cleared to run normally,
  hills included. Less good: I've started a new job, so I can only run three days a week now -- Tuesday, Friday and
  Sunday. Sunday's long run was 18 km in 1:51. Can you rework my week around the new days?

---- the coach with memory Mem0 ----
Hey Maya – great news on the IT‑band and the cleared‑to‑run status! With the new three‑day window (Tue‑Fri‑Sun) we can still hit all the key sessions and protect that knee.

**Your revised week**

| Day | Workout | Details |
|-----|----------|---------|
| **Tuesday** | **Tempo Run** | 7 km total – 1 km easy warm‑up, 5 km at 4:45‑5:00 min/km (85‑90 spm cadence), 1 km cool‑down. Finish with the 5‑minute core circuit (plank, side‑plank, dead‑bug). |
| **Friday** | **Hill/Speed Session** – choose one each week to keep variety and limit total hill load. | **Option A – Hills**: 1 km warm‑up, 2 × 30‑sec hill repe

In [5]:
K.side_by_side("maya-5", with_mem0, from_scratch)

maya-5, 2026-09-29 -- Maya:
  I ran a 10k race on Saturday in 49:30 -- a personal best! I went out far too fast in the first kilometre, as I
  always seem to at race starts, and paid for it at the end. Because of that result I've decided to go for 1:45 at
  the half instead of 1:50. Is that realistic, and how should my next few weeks look?

---- the coach with memory Mem0 ----
**Congrats on the 10 K PB!** 49:30 is a 4:57 min/km average – right on the pace you’ll need to hold for a 1:45 half (≈5:00 min/km).  With your current mileage and the work you’ve already done, the goal is **realistic**, but you’ll need to sharpen a few things in the last three weeks.

---

### 1. What to keep (and why)

| Day | Current session | Why it stays |
|-----|----------------|--------------|
| **Tue** | 7 km total (1 km WU + 5 km tempo + 1 km CD) | Gives you a solid “steady‑state” at goal pace. |
| **Fri** | Hill or 400 m speed work **+** eccentric quad squats & glute work | Keeps leg strength, improves e

In [6]:
K.side_by_side("tom-3", with_mem0, from_scratch)

tom-3, 2026-09-30 -- Tom:
  Quick one: what's my longest run been so far, and how many weeks until my race? I want to plan my taper. And what
  should I eat on race morning, and when?

---- the coach with memory Mem0 ----
**Longest run so far**  
- Your longest completed long‑run is **28 km** (Saturday 12 Sept). The 30 km you had scheduled for 16 Sept was postponed.

**Time to the City Marathon**  
- Today is 30 Sept 2026. The marathon is on 8 Nov 2026 → **5 weeks + 3 days** (≈ 5½ weeks) left.

---

### Taper outline (5 weeks → race day)

| Week | Mon‑Fri (easy/tempo) | Sat (long run) | Sun (recovery) |
|------|----------------------|----------------|----------------|
| **5** (23‑29 Oct) | 8‑10 km total, easy + 1 × 8‑km at marathon pace | **20 km** (steady, take gels) | 5‑6 km easy |
| **4** (30 Oct‑5 Nov) | 6‑8 km total, easy | **12 km** (last “race‑pace” segment 4 km) | 4‑5 km easy |
| **3** (6‑12 Nov) | 5‑6 km easy, short strides | **8 km** (all easy, finish with a quick 2‑km pick‑u

In [7]:
K.side_by_side("maya-6", with_mem0, from_scratch)

maya-6, 2026-10-13 -- Maya:
  Final check before I start tapering. Can you remind me: when is my race, what's my goal time, which days do I run
  now and at what time of day, and how have my long runs built up so far? And any advice for race morning?

---- the coach with memory Mem0 ----
**Race & Goal**  
- **Event:** Riverside Half Marathon – Sunday **Nov 1 2026**  
- **Target finish:** **1 hour 45 minutes** (≈ 4 : 58 min/km)

**Current training pattern**  
- **Days:** **Tuesday, Friday, Sunday**  
- **Time of day:** All runs are **morning sessions** (you switched to pre‑work runs in mid‑September).  

| Day | Typical session |
|-----|-----------------|
| **Tue** | 7 km total – 1 km easy warm‑up, **5 km tempo** at 4:45‑5:00 min/km, 1 km cool‑down |
| **Fri** | Alternating **hill repeats** (4 × 30 s) **or** 400 m speed work + eccentric quad squats & glute work (≈ 6 km total) |
| **Sun** | Long run – building distance, easy pace, finish with a short stride/stride‑out (≈ 16‑20 km) |

**L

## 6 · What each memory holds at the end

Everything Mem0 keeps for Maya, by the check-in it came from:

In [8]:
for row in M.holdings(mem0, "maya"):
    print(row["session"], " ", row["memory"])

2026-09-01   Maya experiences left knee soreness after long downhill stretches
2026-09-01   Maya runs four days a week on Tuesday, Thursday, Saturday, and a long run on Sunday, always in the evening after work
2026-09-01   Maya was given a knee-care plan for downhill soreness focusing on running technique with a cadence of 80-90 spm, strength exercises like eccentric quad squats, glute bridges, and clamshells, plus foam rolling and icing
2026-09-01   Maya was provided a long-run schedule leading up to the October 18, 2026 half marathon, starting with 16 km on September 8 and peaking at 20 km on September 22
2026-09-01   Maya was recommended a weekly mid-week running pattern featuring a Tuesday tempo run, Thursday speed or hill intervals, and a Saturday recovery run with a short core routine
2026-09-01   Maya's longest run before September 2026 was 14 km completed on Sunday, August 30, 2026, in 1:26
2026-09-01   User's name is Maya and she signed up for the Riverside Half Marathon on Su

And the scratch coach's three stores for Maya. Each profile field shows what it said before the last change, which is kept but not shown to the coach:

In [9]:
held = S.holdings("maya")
for field, entry in held["profile"].items():
    print(f"{field:12} {entry['value']}   (since {entry['since']}; before: {entry['was']})")
print()
for run in held["runs"]:
    print(run["on_day"], run["kind"], run["km"], "km", run["minutes"] or "", run["felt"])
print()
for note in held["notes"]:
    print(note["at"], note["text"])

race         Riverside Half Marathon   (since 2026-09-01; before: None)
goal         finish half marathon under 1:45   (since 2026-09-29; before: finish under 1:50)
days         Tue, Thu, Sat, Sun   (since 2026-09-01; before: None)
time_of_day  morning   (since 2026-09-15; before: evening)
injuries     none   (since 2026-09-08; before: left knee soreness after long downhill stretches)

2026-08-27 long 14.0 km  
2026-09-03 long 16.0 km  
2026-09-05 easy 6.0 km  
2026-09-07 easy 6.0 km  
2026-09-12 long 17.0 km  
2026-09-28 race 10.0 km  

2026-09-29 10k PB shows speed; need pacing work for half marathon
2026-09-22 Knee cleared by physio after IT-band tightness; can resume hills.
2026-09-22 Training days changed to Tue, Fri, Sun; Sunday long run 18 km in 1:51.
2026-09-15 knee soreness after hill repeats on 2026-09-10
2026-09-01 Race on 2026-10-18


What the scratch coach was shown before its last reply to Maya:

In [10]:
print(from_scratch["maya-6"]["shown"])

What you know about Maya from earlier check-ins.
Profile:
- race: Riverside Half Marathon (since 2026-09-01)
- goal: finish half marathon under 1:45 (since 2026-09-29)
- days: Tue, Thu, Sat, Sun (since 2026-09-01)
- time_of_day: morning (since 2026-09-15)
- injuries: none (since 2026-09-08)
Runs, oldest first:
- 2026-08-27 long 14 km
- 2026-09-03 long 16 km
- 2026-09-05 easy 6 km
- 2026-09-07 easy 6 km
- 2026-09-12 long 17 km
- 2026-09-28 race 10 km
Notes that bear on this message:
- 2026-09-22 Training days changed to Tue, Fri, Sun; Sunday long run 18 km in 1:51.
- 2026-09-29 10k PB shows speed; need pacing work for half marathon
- 2026-09-01 Race on 2026-10-18


## 7 · What Mem0 brought back

For each check-in: how many memories the search returned, and the meaning score of the most and least alike. Every memory returned went into the coach's prompt.

In [11]:
for row in K.recall_report(with_mem0):
    print(row)
print()
K.show(K.isolation(with_mem0))

{'check-in': 'maya-1', 'recalled': 0, 'most alike': None, 'least alike': None}
{'check-in': 'tom-1', 'recalled': 0, 'most alike': None, 'least alike': None}
{'check-in': 'maya-2', 'recalled': 7, 'most alike': 0.724, 'least alike': 0.558}
{'check-in': 'maya-3', 'recalled': 12, 'most alike': 0.694, 'least alike': 0.602}
{'check-in': 'tom-2', 'recalled': 5, 'most alike': 0.675, 'least alike': 0.585}
{'check-in': 'maya-4', 'recalled': 18, 'most alike': 0.711, 'least alike': 0.547}
{'check-in': 'maya-5', 'recalled': 20, 'most alike': 0.633, 'least alike': 0.526}
{'check-in': 'tom-3', 'recalled': 11, 'most alike': 0.659, 'least alike': 0.577}
{'check-in': 'maya-6', 'recalled': 20, 'most alike': 0.68, 'least alike': 0.542}

{
 "recalled memories": 93,
 "from the other runner": []
}


## 8 · What it cost

In [12]:
K.show(K.cost(with_mem0, from_scratch))

{
 "the Mem0 coach, on Groq": {
  "calls": 9,
  "prompt": 6303,
  "completion": 10159
 },
 "Mem0's model, on Gemini": {
  "calls": 9,
  "prompt": 90517,
  "output (thinking included)": 3138,
  "seconds": 23
 },
 "the scratch coach, on Groq": {
  "calls": 17,
  "prompt": 15071,
  "completion": 9724
 }
}


## 9 · Round 2: each build's writing fixed

Round 1 went wrong for both builds at the same point: **what got written down**.

- **The scratch coach wrote memory as a side job of replying, and wrote it carelessly.** It saved Maya's race without its date, and Tom's too. It never wrote the move to 1 November at all. Her change of days and the knee all-clear went into notes instead of the profile fields that hold them. Two runs got the wrong date, and her 18 km run was never logged. So at her last check-in the profile said 18 October and Tue/Thu/Sat/Sun, and the coach repeated both. Tom, whose race date was never stored, was told it was "about 2 weeks" away. The stores did what they were built to do; they were handed the wrong things.
- **Mem0 stored the coach's own advice as memories.** 10 of Maya's 26 memories are the coach's advice ("Maya was recommended…" or similar), and search returns up to 20 memories every check-in. Its 0.1 threshold filters nothing here: no memory returned scored below 0.52. Twice the coach read a run *it had planned* as one Maya had done ("you've now hit the 20 km peak").

Each build gets one fix, made with its own tools.

**Mem0**: its custom instructions, which every `add` already carries for the date, also say to keep only what the runner says:

```python
# coach_mem0.py
# Round 2. In round 1 Mem0 stored the coach's own advice as memories ("Maya was recommended a 20 km long run on 27
# September"), nearly half of Maya's, and the coach later read a planned run as one she had done. Mem0's custom
# instructions are its own way to say what to keep.
RUNNER_ONLY = (" Extract only what the runner says: their facts, their plans, what they did and how it went. Do not "
               "extract the coach's recommendations, plans or advice -- they are in the coach's replies already, and "
               "a run the coach suggested is not a run that happened.")
```

**From scratch**: writing gets a step of its own, as Mem0 gives it. After the reply, one chapter 1 `extract()` is shown what is on record and the runner's message, and returns the changes as a form. The code routes them through the same `keep` as before. The coach no longer has a tool: it only replies. Today's weekday goes in as well, because "last Sunday" is arithmetic that the date alone leaves to the model.

```python
# coach_scratch.py
RECORD = """You keep a running coach's records about {name}. Today is {today}.

On record now:
{record}

{name}'s message today:
{message}

Return what to change in the record -- only what {name} says, nothing a coach recommends.
- profile: every field the message changes or adds. Write the field's whole new value, complete on its own: a race
  with its name and its date ("Riverside Half Marathon, Sunday 1 November"), every day they run, and so on. It
  replaces what is on record, so leave out a field the message does not change.
- runs: every run the message reports, each dated by working back from today, {today}.
- notes: anything else worth knowing at a later check-in that no profile field holds.
Leave a list empty when nothing applies."""
```

```python
# coach_scratch.py
def write_step(memory, session):
    """Round 2's writing: after the reply, one `extract()` shown the record and the message, its changes routed by
    `keep`. Today's weekday goes in, because "last Sunday" is arithmetic the date alone leaves to the model."""
    from datetime import date

    from llm import extract
    name = R.RUNNERS[session["runner"]]
    today = f"{date.fromisoformat(session['date']):%A} {session['date']}"
    profile, runs = memory.lines()
    record = "\n".join(["Profile:", *(profile or ["- nothing yet"]), "Runs:", *(runs or ["- none logged"])])
    changes = extract(RECORD.format(name=name, today=today, record=record, message=session["message"]), Changes)
    said = memory.keep(session["date"], changes.profile, changes.runs, changes.notes)
    return {**changes.model_dump(), "said": said}
```

Both rounds run on fresh stores, so round 1's memories stay as they were, for comparison.

In [2]:
mem0_2 = M.memory(M.ROUND2)
with_mem0_2, finished = M.run(mem0_2, path=M.ANSWERS2, instructions=M.RUNNER_ONLY)
print("\nall nine done" if finished else "\nstopped -- run this cell again to carry on")

  maya-1: recalled 0, stored 4


  tom-1: recalled 0, stored 5


  maya-2: recalled 4, stored 2


  maya-3: recalled 6, stored 4


  tom-2: recalled 5, stored 3


  maya-4: recalled 10, stored 3


  maya-5: recalled 13, stored 2


  tom-3: recalled 8, stored 0


  maya-6: recalled 15, stored 0

all nine done


In [3]:
from_scratch_2, finished = S.run(root=S.ROUND2, path=S.ANSWERS2, write="step")
print("\nall nine done" if finished else "\nstopped -- run this cell again to carry on")

  maya-1: kept 1 time(s), 2 calls, 3,060 tokens


  tom-1: kept 1 time(s), 2 calls, 2,386 tokens


  maya-2: kept 1 time(s), 2 calls, 3,197 tokens


  maya-3: kept 1 time(s), 2 calls, 3,567 tokens


  tom-2: kept 1 time(s), 2 calls, 2,804 tokens


  maya-4: kept 1 time(s), 2 calls, 3,125 tokens


  maya-5: kept 1 time(s), 2 calls, 3,429 tokens


  tom-3: kept 1 time(s), 2 calls, 2,588 tokens


  maya-6: kept 1 time(s), 2 calls, 2,541 tokens

all nine done


## 10 · Round 2, side by side

In [3]:
K.side_by_side("maya-2", with_mem0_2, from_scratch_2)

maya-2, 2026-09-08 -- Maya:
  Sunday's long run was 16 km in 1:38 -- felt good, and the knee was fine. Tuesday and Thursday were easy 6 km runs.
  Should I add a tempo run, and if so, on which day?

---- the coach with memory Mem0 ----
Great job on the 16 km long run—16 km in 1:38 with a clean knee is a solid step toward that sub‑1:50 half‑marathon goal. Your easy 6 km runs on Tue/Thu look good for recovery, too.

**Tempo suggestion**

| Day | Session | How to fit it in |
|-----|---------|------------------|
| **Thursday** | **Tempo** (after a 1‑km easy jog) | 5 km at half‑marathon goal pace ≈ 5:10–5:15 / km, then 1 km cool‑down. Keep the total Thursday mileage around 7 km so you stay easy overall. |
| **Saturday** | Keep as easy/stride day | 6 km easy + 4–6 × 100 m strides to keep leg turnover sharp without extra fatigue before Sunday’s long. |

**Why Thursday?**  
- It gives you a solid mid‑week quality effort without cramming it right before the long run.  
- You still have a recove

In [4]:
K.side_by_side("maya-4", with_mem0_2, from_scratch_2)

maya-4, 2026-09-22 -- Maya:
  Good news: I saw a physio and my knee is fine -- it was a tight IT band, and I've been cleared to run normally,
  hills included. Less good: I've started a new job, so I can only run three days a week now -- Tuesday, Friday and
  Sunday. Sunday's long run was 18 km in 1:51. Can you rework my week around the new days?

---- the coach with memory Mem0 ----
Hey Maya – great news about the IT‑band and getting cleared for hills! With the new three‑day schedule we can still hit the quality sessions you need for a sub‑1:50 half‑marathon while keeping the load manageable for your knee.

**This week (Tue / Fri / Sun)**  

| Day | Workout | Why it helps | Approx. km |
|-----|----------|--------------|------------|
| **Tuesday** | **Easy + Strides** – 8 km total: 6 km easy (≈6:30‑6:45 / km) + 4 × 100 m strides (slow jog back) | Keeps aerobic base, promotes good form, flushes out any residual tightness | 8 |
| **Friday** | **Hill Repeats** – 1 km warm‑up, 6 × 90 s uph

In [5]:
K.side_by_side("maya-5", with_mem0_2, from_scratch_2)

maya-5, 2026-09-29 -- Maya:
  I ran a 10k race on Saturday in 49:30 -- a personal best! I went out far too fast in the first kilometre, as I
  always seem to at race starts, and paid for it at the end. Because of that result I've decided to go for 1:45 at
  the half instead of 1:50. Is that realistic, and how should my next few weeks look?

---- the coach with memory Mem0 ----
**Congrats on the 10 K PB, Maya!** A 49:30 finish (≈4:57 km⁻¹) shows you’re already right on the pace you need for a 1:45 half (≈4:58 km⁻¹). The big question is turning that speed into sustained endurance while keeping the IT‑band happy.

---

### Is 1:45 realistic?
- **Current fitness:** 10 K at 4:57 km⁻¹ → you have the speed.
- **Long‑run pace:** 18 km in 1:51 = 5:12 km⁻¹, about 15 s slower than goal.
- **Gap to close:** ≈15 s/km over the second half of a race. With a few targeted workouts and a proper taper, a 1:45 is **within reach**.

---

## 3‑day‑a‑week plan (Tue / Fri / Sun) – next 5 weeks

| Week | Tue –

In [6]:
K.side_by_side("tom-3", with_mem0_2, from_scratch_2)

tom-3, 2026-09-30 -- Tom:
  Quick one: what's my longest run been so far, and how many weeks until my race? I want to plan my taper. And what
  should I eat on race morning, and when?

---- the coach with memory Mem0 ----
**Longest training run so far:**  
- **28 km** on Saturday 12 Sept (the run where you switched to energy gels).

**Time to the City Marathon (Nov 8, 2026):**  
- From today (30 Sept) it’s **5 weeks + 2 days** – essentially a 5‑week taper window.

---

### Quick taper outline (5 weeks → race)

| Week | Goal | Mileage (≈ % of peak week) | Key workouts |
|------|------|----------------------------|--------------|
| **5** (now) | Begin dropping volume | 80 % of peak (≈ 30‑32 km total) | Keep the Saturday long run at **20 km** (easy pace). |
| **4** | Sharpen | 60 % (≈ 22‑24 km) | Saturday **15 km** easy + a short 5 km at goal marathon pace. |
| **3** | Light & fast | 45 % (≈ 16‑18 km) | Saturday **12 km** with 2 × 3 km at race pace. |
| **2** | Race‑ready | 30 % (≈ 10‑12 

In [7]:
K.side_by_side("maya-6", with_mem0_2, from_scratch_2)

maya-6, 2026-10-13 -- Maya:
  Final check before I start tapering. Can you remind me: when is my race, what's my goal time, which days do I run
  now and at what time of day, and how have my long runs built up so far? And any advice for race morning?

---- the coach with memory Mem0 ----
**Race & Goal**  
- **Riverside Half Marathon:** **Sunday, Nov 1 2026**  
- **Target finish:** **1 hour 45 minutes** (≈ 5 min /km or 8 min /mile)

**Current weekly schedule** (all **morning runs before work**)  
| Day | Run type | Approx. duration |
|-----|----------|------------------|
| **Tuesday** | Easy/tempo (6‑8 km) | 30‑40 min |
| **Friday**   | Hill repeats or speed work (5‑7 km) | 35‑45 min |
| **Sunday**   | Long run (currently 18 km, tapering soon) | 1 h 45‑2 h |

**Long‑run progression so far**  

| Date | Distance | Time | Notes |
|------|----------|------|-------|
| Aug 30 (Sun) | 14 km | 1:26 | First long run of the block |
| Sep 6 (Sun)  | 16 km | 1:38 | Felt good, no knee issues |
| Se

## 11 · What each memory holds after round 2

In [8]:
for row in M.holdings(mem0_2, "maya"):
    print(row["session"], " ", row["memory"])

2026-09-01   Maya experiences left knee soreness after long downhill stretches
2026-09-01   Maya runs four days a week on Tuesday, Thursday, Saturday, and a long run on Sunday, always in the evening after work
2026-09-01   Maya's longest run so far was 14 km on Sunday, August 30, 2026, completed in 1:26
2026-09-01   User's name is Maya and she has signed up for the Riverside Half Marathon on Sunday, October 18, 2026, with a goal finish time under 1:50
2026-09-08   Maya completed a 16 km long run in 1:38 on Sunday, September 6, 2026, felt good, and experienced no knee issues
2026-09-08   Maya completed easy 6 km runs on Tuesday, September 8, 2026, and Thursday, September 10, 2026
2026-09-15   Maya completed a 17 km long run in 1:45 on Sunday, September 13, 2026
2026-09-15   Maya experienced left knee soreness after doing hill repeats on Thursday, September 10, 2026
2026-09-15   Maya switched her running schedule from evenings to mornings before work because evenings were getting too bus

In [9]:
held_2 = S.holdings("maya", S.ROUND2)
for field, entry in held_2["profile"].items():
    print(f"{field:12} {entry['value']}   (since {entry['since']}; before: {entry['was']})")
print()
for run in held_2["runs"]:
    print(run["on_day"], run["kind"], run["km"], "km", run["minutes"] or "", run["felt"])
print()
for note in held_2["notes"]:
    print(note["at"], note["text"])
print("\nTom:", {field: entry["value"] for field, entry in S.holdings("tom", S.ROUND2)["profile"].items()})

race         Riverside Half Marathon, Sunday 1 November   (since 2026-09-15; before: Riverside Half Marathon, Sunday 18 October)
goal         under 1:45   (since 2026-09-29; before: under 1:50)
days         Tuesday, Friday, Sunday   (since 2026-09-22; before: Tuesday, Thursday, Saturday, Sunday)
time_of_day  morning   (since 2026-09-15; before: evening)
injuries     knee fine (tight IT band resolved)   (since 2026-09-22; before: knee sore)

2026-08-29 long 14.0 km 86.0 
2026-09-03 easy 6.0 km  
2026-09-04 long 16.0 km 98.0 felt good
2026-09-08 easy 6.0 km  
2026-09-13 long 17.0 km 105.0 
2026-09-20 long 18.0 km 111.0 
2026-09-26 race 10.0 km 49.5 personal best

2026-10-13 Maya is preparing to start tapering before her Riverside Half Marathon on Sunday 1 November.
2026-09-29 She went out far too fast in the first kilometre during the 10k race, paying for it at the end.
2026-09-22 Started new job, limiting runs to three days per week
2026-09-08 Maya asked whether to add a tempo run and w

In [10]:
for row in K.recall_report(with_mem0_2):
    print(row)
print()
K.show(K.isolation(with_mem0_2))
print()
K.show(K.cost(with_mem0_2, from_scratch_2))

{'check-in': 'maya-1', 'recalled': 0, 'most alike': None, 'least alike': None}
{'check-in': 'tom-1', 'recalled': 0, 'most alike': None, 'least alike': None}
{'check-in': 'maya-2', 'recalled': 4, 'most alike': 0.622, 'least alike': 0.556}
{'check-in': 'maya-3', 'recalled': 6, 'most alike': 0.671, 'least alike': 0.602}
{'check-in': 'tom-2', 'recalled': 5, 'most alike': 0.664, 'least alike': 0.516}
{'check-in': 'maya-4', 'recalled': 10, 'most alike': 0.656, 'least alike': 0.524}
{'check-in': 'maya-5', 'recalled': 13, 'most alike': 0.604, 'least alike': 0.511}
{'check-in': 'tom-3', 'recalled': 8, 'most alike': 0.656, 'least alike': 0.508}
{'check-in': 'maya-6', 'recalled': 15, 'most alike': 0.655, 'least alike': 0.505}

{
 "recalled memories": 61,
 "from the other runner": []
}

{
 "the Mem0 coach, on Groq": {
  "calls": 9,
  "prompt": 4268,
  "completion": 9789
 },
 "Mem0's model, on Gemini": {
  "calls": 9,
  "prompt": 89253,
  "output (thinking included)": 1447,
  "seconds": 23
 },
 "th

## 12 · Round 3: the scratch build's writing, checked

Round 2's write step got every named fact right at the last check-in, but it still wrote three kinds of wrong fact. A profile field shows whatever it holds as a plain, current fact, so each one reached the coach:

- **A field left out.** "Gels worked" went into a run's note, so `food` kept "water only", and the coach told Tom he had done well on water alone. The profile was an *optional list*: a field the model did not think of counted as unchanged, silently.
- **A value made up.** Tom's "five days a week" was saved as "Monday, Tuesday, Wednesday, Thursday, Friday". A form invites filling in.
- **A miss with nothing beside it.** A replaced value leaves no trace, so when a change is missed the coach cannot see that anything changed.

The fixes:
1. **A slot for every field.** The form has one required slot per field, answered "unchanged" or with the new value, so `food` has to be decided on when the message talks about gels. It can still be decided wrongly; it can no longer be skipped by not thinking of it.
2. **Every change checked against the runner's words**, in code, before it is kept: chapter 3's `verified()` idea, checking the result rather than the report of it. Each change carries a `quote`, which must be the message's own words in order. Every day, month and number in the value must be in that quote. Refused changes are not kept, and are recorded with the reason.
3. **The old value stays in view.** Each field is shown with what it said before and since when. Each check-in's message is also kept in the journal, dated, so `Recall` can bring back what the runner actually said.

Mem0 is not re-run: its round 2 is what round 3 is compared with.

```python
# coach_scratch.py
CHECKED = """You keep a running coach's records about {name}. Today is {today}.

On record now:
{record}

{name}'s message today:
{message}

Fill in the record form -- only what {name} says, nothing a coach recommends.
- profile: answer EVERY field. "unchanged" if the message does not change it; otherwise its whole new value,
  complete on its own (a race with its name and its date). With each new value, copy the words of the message it
  comes from into `quote`, exactly. The value must say nothing the quote does not: no day, date or number that is
  not in the quote.
- runs: every run the message reports, each dated by working back from today, {today}, with its words in `quote`.
- notes: anything else worth knowing at a later check-in that no profile field holds, with its words in `quote`."""
```

```python
# coach_scratch.py
def unquoted(value, quote, message):
    """Why a change is refused, or None. The quote must be the message's own words, in order; and every day, month
    and number in the value must be in the quote -- the parts a model fills in when it fills a form."""
    said, words = _words(quote), _words(message)
    if not said:
        return "no quote"
    if not any(words[i:i + len(said)] == said for i in range(len(words) - len(said) + 1)):
        return "the quote is not in the message"
    claimed = [w for w in _words(value) if w in _NAMES or re.fullmatch(r"\d+(:\d+)?", w)]
    missing = [w for w in dict.fromkeys(claimed) if w not in said]
    return f"not in the quote: {', '.join(missing)}" if missing else None

def checked_step(memory, session):
    """Round 3's writing: a form with a slot for every field; each change checked against the message's own words
    before it is kept, the refused ones recorded with why; and the message itself kept in the journal, dated."""
    from datetime import date

    from llm import extract
    name, message = R.RUNNERS[session["runner"]], session["message"]
    today = f"{date.fromisoformat(session['date']):%A} {session['date']}"
    profile, runs = memory.lines()
    record = "\n".join(["Profile:", *(profile or ["- nothing yet"]), "Runs:", *(runs or ["- none logged"])])
    form = extract(CHECKED.format(name=name, today=today, record=record, message=message), Form)
    refused, changes, kept_runs, kept_notes = [], [], [], []
    for field in PROFILE:
        slot = getattr(form.profile, field)
        if slot.value.strip().strip('"').lower() in ("unchanged", ""):
            continue
        why = unquoted(slot.value, slot.quote, message)
        if why:
            refused.append({"field": field, "value": slot.value, "quote": slot.quote, "why": why})
        else:
            changes.append(Change(field=field, value=slot.value))
    for run in form.runs:
        why = unquoted(f"{run.km:g}", run.quote, message)
        (refused.append({"run": run.model_dump(), "why": why}) if why else kept_runs.append(run))
    for note in form.notes:
        why = unquoted("", note.quote, message)
        (refused.append({"note": note.text, "quote": note.quote, "why": why}) if why else kept_notes.append(note.text))
    said = memory.keep(session["date"], changes, kept_runs, kept_notes)
    entry = f"{name} said: {message}"
    if entry not in {row["text"] for row in memory.notes.recent(limit=1000)}:
        memory.notes.write([(session["date"], memory.runner, "runner said", entry)], now=session["date"])
    return {"form": form.model_dump(), "refused": refused, "said": said}
```

In [2]:
from_scratch_3, finished = S.run(root=S.ROUND3, path=S.ANSWERS3, write="checked")
print("\nall nine done" if finished else "\nstopped -- run this cell again to carry on")

  maya-1: kept 1 time(s), 2 calls, 3,636 tokens


  tom-1: kept 1 time(s), 2 calls, 3,340 tokens


  maya-2: kept 1 time(s), 2 calls, 3,850 tokens


  maya-3: kept 1 time(s), 2 calls, 4,391 tokens


  tom-2: kept 1 time(s), 2 calls, 3,816 tokens


  maya-4: kept 1 time(s), 2 calls, 4,955 tokens


  maya-5: kept 1 time(s), 2 calls, 5,779 tokens


  tom-3: kept 1 time(s), 2 calls, 4,189 tokens


  maya-6: kept 1 time(s), 2 calls, 4,172 tokens

all nine done


What the check refused, and why:

In [3]:
for sid, record in from_scratch_3.items():
    for refused in record["kept"][0]["refused"]:
        print(sid, refused)

tom-1 {'field': 'food', 'value': 'vegetarian, water alone', 'quote': "I'm vegetarian, and so far I've done the long runs on water alone", 'why': 'the quote is not in the message'}


## 13 · Round 3 against Mem0's round 2

In [4]:
K.side_by_side("maya-2", with_mem0_2, from_scratch_3)

maya-2, 2026-09-08 -- Maya:
  Sunday's long run was 16 km in 1:38 -- felt good, and the knee was fine. Tuesday and Thursday were easy 6 km runs.
  Should I add a tempo run, and if so, on which day?

---- the coach with memory Mem0 ----
Great job on the 16 km long run—16 km in 1:38 with a clean knee is a solid step toward that sub‑1:50 half‑marathon goal. Your easy 6 km runs on Tue/Thu look good for recovery, too.

**Tempo suggestion**

| Day | Session | How to fit it in |
|-----|---------|------------------|
| **Thursday** | **Tempo** (after a 1‑km easy jog) | 5 km at half‑marathon goal pace ≈ 5:10–5:15 / km, then 1 km cool‑down. Keep the total Thursday mileage around 7 km so you stay easy overall. |
| **Saturday** | Keep as easy/stride day | 6 km easy + 4–6 × 100 m strides to keep leg turnover sharp without extra fatigue before Sunday’s long. |

**Why Thursday?**  
- It gives you a solid mid‑week quality effort without cramming it right before the long run.  
- You still have a recove

In [5]:
K.side_by_side("maya-4", with_mem0_2, from_scratch_3)

maya-4, 2026-09-22 -- Maya:
  Good news: I saw a physio and my knee is fine -- it was a tight IT band, and I've been cleared to run normally,
  hills included. Less good: I've started a new job, so I can only run three days a week now -- Tuesday, Friday and
  Sunday. Sunday's long run was 18 km in 1:51. Can you rework my week around the new days?

---- the coach with memory Mem0 ----
Hey Maya – great news about the IT‑band and getting cleared for hills! With the new three‑day schedule we can still hit the quality sessions you need for a sub‑1:50 half‑marathon while keeping the load manageable for your knee.

**This week (Tue / Fri / Sun)**  

| Day | Workout | Why it helps | Approx. km |
|-----|----------|--------------|------------|
| **Tuesday** | **Easy + Strides** – 8 km total: 6 km easy (≈6:30‑6:45 / km) + 4 × 100 m strides (slow jog back) | Keeps aerobic base, promotes good form, flushes out any residual tightness | 8 |
| **Friday** | **Hill Repeats** – 1 km warm‑up, 6 × 90 s uph

In [6]:
K.side_by_side("maya-5", with_mem0_2, from_scratch_3)

maya-5, 2026-09-29 -- Maya:
  I ran a 10k race on Saturday in 49:30 -- a personal best! I went out far too fast in the first kilometre, as I
  always seem to at race starts, and paid for it at the end. Because of that result I've decided to go for 1:45 at
  the half instead of 1:50. Is that realistic, and how should my next few weeks look?

---- the coach with memory Mem0 ----
**Congrats on the 10 K PB, Maya!** A 49:30 finish (≈4:57 km⁻¹) shows you’re already right on the pace you need for a 1:45 half (≈4:58 km⁻¹). The big question is turning that speed into sustained endurance while keeping the IT‑band happy.

---

### Is 1:45 realistic?
- **Current fitness:** 10 K at 4:57 km⁻¹ → you have the speed.
- **Long‑run pace:** 18 km in 1:51 = 5:12 km⁻¹, about 15 s slower than goal.
- **Gap to close:** ≈15 s/km over the second half of a race. With a few targeted workouts and a proper taper, a 1:45 is **within reach**.

---

## 3‑day‑a‑week plan (Tue / Fri / Sun) – next 5 weeks

| Week | Tue –

In [7]:
K.side_by_side("tom-3", with_mem0_2, from_scratch_3)

tom-3, 2026-09-30 -- Tom:
  Quick one: what's my longest run been so far, and how many weeks until my race? I want to plan my taper. And what
  should I eat on race morning, and when?

---- the coach with memory Mem0 ----
**Longest training run so far:**  
- **28 km** on Saturday 12 Sept (the run where you switched to energy gels).

**Time to the City Marathon (Nov 8, 2026):**  
- From today (30 Sept) it’s **5 weeks + 2 days** – essentially a 5‑week taper window.

---

### Quick taper outline (5 weeks → race)

| Week | Goal | Mileage (≈ % of peak week) | Key workouts |
|------|------|----------------------------|--------------|
| **5** (now) | Begin dropping volume | 80 % of peak (≈ 30‑32 km total) | Keep the Saturday long run at **20 km** (easy pace). |
| **4** | Sharpen | 60 % (≈ 22‑24 km) | Saturday **15 km** easy + a short 5 km at goal marathon pace. |
| **3** | Light & fast | 45 % (≈ 16‑18 km) | Saturday **12 km** with 2 × 3 km at race pace. |
| **2** | Race‑ready | 30 % (≈ 10‑12 

In [8]:
K.side_by_side("maya-6", with_mem0_2, from_scratch_3)

maya-6, 2026-10-13 -- Maya:
  Final check before I start tapering. Can you remind me: when is my race, what's my goal time, which days do I run
  now and at what time of day, and how have my long runs built up so far? And any advice for race morning?

---- the coach with memory Mem0 ----
**Race & Goal**  
- **Riverside Half Marathon:** **Sunday, Nov 1 2026**  
- **Target finish:** **1 hour 45 minutes** (≈ 5 min /km or 8 min /mile)

**Current weekly schedule** (all **morning runs before work**)  
| Day | Run type | Approx. duration |
|-----|----------|------------------|
| **Tuesday** | Easy/tempo (6‑8 km) | 30‑40 min |
| **Friday**   | Hill repeats or speed work (5‑7 km) | 35‑45 min |
| **Sunday**   | Long run (currently 18 km, tapering soon) | 1 h 45‑2 h |

**Long‑run progression so far**  

| Date | Distance | Time | Notes |
|------|----------|------|-------|
| Aug 30 (Sun) | 14 km | 1:26 | First long run of the block |
| Sep 6 (Sun)  | 16 km | 1:38 | Felt good, no knee issues |
| Se

In [9]:
held_3 = S.holdings("maya", S.ROUND3)
for field, entry in held_3["profile"].items():
    print(f"{field:12} {entry['value']}   (since {entry['since']}; before: {entry['was']})")
print()
for run in held_3["runs"]:
    print(run["on_day"], run["kind"], run["km"], "km", run["minutes"] or "", run["felt"])
print("\nTom:", {field: entry["value"] for field, entry in S.holdings("tom", S.ROUND3)["profile"].items()})
print()
K.show(K.cost(with_mem0_2, from_scratch_3)["the scratch coach, on Groq"])

race         Riverside Half on Sunday 1 November   (since 2026-09-15; before: Riverside Half Marathon on Sunday 18 October)
goal         go for 1:45 at the half   (since 2026-09-29; before: finish under 1:50)
days         Tuesday, Friday and Sunday   (since 2026-09-22; before: Tuesday, Thursday, Saturday, Sunday)
time_of_day  morning   (since 2026-09-15; before: evening)
injuries     my knee is fine   (since 2026-09-22; before: knee sore after hill repeats on Thursday)

2026-08-28 long 14.0 km 86.0 
2026-09-03 easy 6.0 km  
2026-09-05 long 16.0 km 98.0 
2026-09-08 easy 6.0 km  
2026-09-13 long 17.0 km 105.0 
2026-09-19 long 18.0 km 111.0 
2026-09-26 race 10.0 km 49.5 

Tom: {'race': 'City Marathon on Sunday 8 November', 'days': 'five days a week', 'injuries': 'right hamstring tight; stomach cramps for first 5 km', 'food': 'gels'}

{
 "calls": 18,
 "prompt": 15394,
 "completion": 22734
}


## 14 · Round 4: three of round 3's own choices undone, and dates worked out in code

Round 3's fixes worked on what they were aimed at, but three of its own choices were mistakes:

- **One line held two facts.** `food` held both diet and race fuel, and a new value replaces the whole line, so writing "gels" erased "vegetarian".
- **The check tested copying, not facts.** A quote had to match the message word for word, and one changed word ("the" for "my") threw a correct fact away.
- **Whole messages went in with the notes.** A whole message covers many topics, so it matches almost any question. Three of them took all of `Recall`'s three places, and the short note about starting too fast never reached Maya's last check-in.

Round 4 also takes on what no round had fixed: the model working out dates ("last Sunday") itself.

The general parts went into chapter 4, because they are about writing memory down, not about running:
- **`Profile`** (in `stores.py`): named facts that keep their previous value and dates, with fields that can be added.
- **`writing.py`:**
  - `profile_form`, a slot for every field and room to add one;
  - `unsupported`, every day, date, number and name in a value must be in the message or already on record;
  - `day_of`, the runner's words turned into a date;
  - `fill`, extract, check, and send back once with the reasons instead of dropping.

What stays here is the coach's own:
- the fields, now with diet and race fuel apart;
- the runs, given in the runner's words for the day and the time;
- `minutes_of` for running times;
- notes of one fact each, and no whole messages.

```python
# writing.py
def unsupported(value, source):
    """The days, dates, numbers and names in `value` that `source` never says, or [].

    Compared by what they mean, not how they are written: "5" is "five", and "2026-11-08", "8 November 2026" and
    "Nov 8" are one day (a year beside a date that matches is part of it). Measured in `06_mem0`: compared as
    written, "8 November" saved as "2026-11-08" was refused, and a runner's race was never stored.

    A name is a capitalised word that is not the first word of the value or of a clause in it -- "Riverside Half
    Marathon, Sunday 1 November" claims Half, Marathon, Sunday and the date. Pass the record as part of the source
    as well as the new message, so a value may repeat what is already known.
    """
    said_dates = {when for _, when in dates(source)}
    missing_dates, rest = [], value
    for (start, end), when in reversed(dates(value)):
        if when not in said_dates:
            missing_dates.append(value[start:end].lower())
        rest = rest[:start] + " " + rest[end:]
    # the source's number words count as digits, so a "5" in the value matches "five"; a number word in the
    # value is not a claim, as before -- only digits are
    value, source = rest, as_digits(source)
    said = set(words(source))
    singular = lambda w: w[:-1] if w.endswith("s") and not w.endswith("ss") else w      # "Sundays" is "Sunday"
    said |= {singular(w) for w in said}
    claimed = [w for w in words(value) if singular(w) in WEEKDAYS or w in MONTHS or re.fullmatch(r"\d+(:\d+)?", w)]
    for clause in re.split(r"[,.;:()\-–—]", value):
        found = re.findall(r"[A-Za-z]+", clause)
        claimed += [w.lower() for w in found[1:] if w[0].isupper()]
    return list(reversed(missing_dates)) + [w for w in dict.fromkeys(claimed) if w not in said and singular(w) not in said]

def fill(prompt, form, problems, extract=None):
    """Extract `form`; if `problems(filled)` finds any, send it back once with the reasons; return the form, what is
    still wrong, and what was sent back. Callers keep what is not in the returned problems."""
    if extract is None:
        from llm import extract
    filled = extract(prompt, form)
    found, sent_back = problems(filled), []
    # a problem marked "drop" is an item that should simply not be kept -- an event copied from the record, say --
    # and asking again costs a whole call to remove what code can remove. Only the others are worth a second try.
    fixable = [p for p in found if not p.get("drop")]
    if fixable:
        sent_back = [f"{p['what']} -- {p['why']}" for p in fixable]
        filled = extract(prompt + "\n\nYour last form had these problems. Fix them, writing only what the message or "
                         "the record says:\n" + "\n".join(f"- {s}" for s in sent_back), form)
        found = problems(filled)
    return filled, found, sent_back
```

```python
# coach_scratch.py
FIELDS4 = {"race": "the race they are training for, and its date",
           "goal": "their goal time",
           "days": "the days of the week they run",
           "time_of_day": "when in the day they run",
           "injuries": "anything that hurts, and how it stands now",
           "diet": "what they eat in general, e.g. vegetarian",
           "fuel": "what they eat and drink before and during runs and races"}
```

```python
# coach_scratch.py
def card_step(memory, session):
    """Round 4's writing, on chapter 4's `writing`: one form with a slot for every field the runner has and room to
    add one; every day, date, number and name checked against the message and the record; anything that fails sent
    back once with the reason; the runner's words for days and times turned into dates and minutes by code."""
    from datetime import date

    import writing as W
    name, message, today = R.RUNNERS[session["runner"]], session["message"], session["date"]
    fields = memory.profile.fields()
    profile, runs = memory.lines()
    record = "\n".join(["Profile:", *(profile or ["- nothing yet"]), "Runs:", *(runs or ["- none logged"])])
    source = message + "\n" + record
    form = W.profile_form(fields, runs=(list[SaidRun], Field(default_factory=list)),
                          notes=(list[str], Field(default_factory=list)))

    def problems(card):
        found = W.profile_problems(card, fields, source)
        for i, run in enumerate(card.runs):
            if W.day_of(run.when, today) is None:
                found.append({"key": ("run", i), "what": f"run '{run.when}'", "why": "no day can be worked out from these words"})
            elif f"{run.km:g}" not in W.words(message):
                found.append({"key": ("run", i), "what": f"run '{run.when}' {run.km:g} km", "why": "the distance is not in the message"})
        for i, note in enumerate(card.notes):
            if bad := W.unsupported(note, source):
                found.append({"key": ("note", i), "what": f"note '{note}'", "why": "not said: " + ", ".join(bad)})
        return found

    prompt = CARD.format(name=name, today=f"{date.fromisoformat(today):%A} {today}", record=record, message=message)
    card, still, sent_back = W.fill(prompt, form, problems)
    refused = {p["key"] for p in still}
    changes = [(f, v) for f, v in W.changed(card, fields) if ("profile", f) not in refused]
    for new in card.new_fields:
        key = re.sub(r"\W+", "_", new.name.lower()).strip("_")
        if key and key not in fields and ("profile", key) not in refused:
            memory.profile.add_field(key, new.about)
    kept_runs = [Run(on=W.day_of(r.when, today), kind=r.kind, km=r.km, minutes=minutes_of(r.time), felt=r.felt)
                 for i, r in enumerate(card.runs) if ("run", i) not in refused]
    kept_notes = [n for i, n in enumerate(card.notes) if ("note", i) not in refused]
    said = memory.keep(today, changes, kept_runs, kept_notes)
    return {"card": card.model_dump(), "sent back": sent_back,
            "refused": [{k: v for k, v in p.items() if k != "key"} for p in still], "said": said}
```

In [2]:
from_scratch_4, finished = S.run(root=S.ROUND4, path=S.ANSWERS4, write="card")
print("\nall nine done" if finished else "\nstopped -- run this cell again to carry on")

  maya-1: kept 1 time(s), 2 calls, 3,493 tokens


  tom-1: kept 1 time(s), 3 calls, 4,878 tokens


  maya-2: kept 1 time(s), 2 calls, 3,799 tokens


  maya-3: kept 1 time(s), 2 calls, 4,539 tokens


  tom-2: kept 1 time(s), 2 calls, 3,717 tokens


  maya-4: kept 1 time(s), 2 calls, 4,454 tokens


  maya-5: kept 1 time(s), 3 calls, 6,765 tokens


  tom-3: kept 1 time(s), 3 calls, 5,572 tokens


  maya-6: kept 1 time(s), 2 calls, 3,254 tokens

all nine done


What was sent back to the model, and what was still refused after its second try:

In [3]:
for sid, record in from_scratch_4.items():
    step = record["kept"][0]
    for line in step["sent back"]:
        print(sid, "sent back:", line)
    for refused in step["refused"]:
        print(sid, "REFUSED:  ", refused)

tom-1 sent back: note 'Tom is training for his first marathon, the City Marathon on 8 November 2026.' -- not said: 2026
tom-3 sent back: note 'There are about 5 weeks until the City Marathon on 2026-11-08.' -- not said: 11


## 15 · Round 4 against Mem0's round 2

In [4]:
K.side_by_side("maya-2", with_mem0_2, from_scratch_4)

maya-2, 2026-09-08 -- Maya:
  Sunday's long run was 16 km in 1:38 -- felt good, and the knee was fine. Tuesday and Thursday were easy 6 km runs.
  Should I add a tempo run, and if so, on which day?

---- the coach with memory Mem0 ----
Great job on the 16 km long run—16 km in 1:38 with a clean knee is a solid step toward that sub‑1:50 half‑marathon goal. Your easy 6 km runs on Tue/Thu look good for recovery, too.

**Tempo suggestion**

| Day | Session | How to fit it in |
|-----|---------|------------------|
| **Thursday** | **Tempo** (after a 1‑km easy jog) | 5 km at half‑marathon goal pace ≈ 5:10–5:15 / km, then 1 km cool‑down. Keep the total Thursday mileage around 7 km so you stay easy overall. |
| **Saturday** | Keep as easy/stride day | 6 km easy + 4–6 × 100 m strides to keep leg turnover sharp without extra fatigue before Sunday’s long. |

**Why Thursday?**  
- It gives you a solid mid‑week quality effort without cramming it right before the long run.  
- You still have a recove

In [5]:
K.side_by_side("maya-4", with_mem0_2, from_scratch_4)

maya-4, 2026-09-22 -- Maya:
  Good news: I saw a physio and my knee is fine -- it was a tight IT band, and I've been cleared to run normally,
  hills included. Less good: I've started a new job, so I can only run three days a week now -- Tuesday, Friday and
  Sunday. Sunday's long run was 18 km in 1:51. Can you rework my week around the new days?

---- the coach with memory Mem0 ----
Hey Maya – great news about the IT‑band and getting cleared for hills! With the new three‑day schedule we can still hit the quality sessions you need for a sub‑1:50 half‑marathon while keeping the load manageable for your knee.

**This week (Tue / Fri / Sun)**  

| Day | Workout | Why it helps | Approx. km |
|-----|----------|--------------|------------|
| **Tuesday** | **Easy + Strides** – 8 km total: 6 km easy (≈6:30‑6:45 / km) + 4 × 100 m strides (slow jog back) | Keeps aerobic base, promotes good form, flushes out any residual tightness | 8 |
| **Friday** | **Hill Repeats** – 1 km warm‑up, 6 × 90 s uph

In [6]:
K.side_by_side("maya-5", with_mem0_2, from_scratch_4)

maya-5, 2026-09-29 -- Maya:
  I ran a 10k race on Saturday in 49:30 -- a personal best! I went out far too fast in the first kilometre, as I
  always seem to at race starts, and paid for it at the end. Because of that result I've decided to go for 1:45 at
  the half instead of 1:50. Is that realistic, and how should my next few weeks look?

---- the coach with memory Mem0 ----
**Congrats on the 10 K PB, Maya!** A 49:30 finish (≈4:57 km⁻¹) shows you’re already right on the pace you need for a 1:45 half (≈4:58 km⁻¹). The big question is turning that speed into sustained endurance while keeping the IT‑band happy.

---

### Is 1:45 realistic?
- **Current fitness:** 10 K at 4:57 km⁻¹ → you have the speed.
- **Long‑run pace:** 18 km in 1:51 = 5:12 km⁻¹, about 15 s slower than goal.
- **Gap to close:** ≈15 s/km over the second half of a race. With a few targeted workouts and a proper taper, a 1:45 is **within reach**.

---

## 3‑day‑a‑week plan (Tue / Fri / Sun) – next 5 weeks

| Week | Tue –

In [7]:
K.side_by_side("tom-3", with_mem0_2, from_scratch_4)

tom-3, 2026-09-30 -- Tom:
  Quick one: what's my longest run been so far, and how many weeks until my race? I want to plan my taper. And what
  should I eat on race morning, and when?

---- the coach with memory Mem0 ----
**Longest training run so far:**  
- **28 km** on Saturday 12 Sept (the run where you switched to energy gels).

**Time to the City Marathon (Nov 8, 2026):**  
- From today (30 Sept) it’s **5 weeks + 2 days** – essentially a 5‑week taper window.

---

### Quick taper outline (5 weeks → race)

| Week | Goal | Mileage (≈ % of peak week) | Key workouts |
|------|------|----------------------------|--------------|
| **5** (now) | Begin dropping volume | 80 % of peak (≈ 30‑32 km total) | Keep the Saturday long run at **20 km** (easy pace). |
| **4** | Sharpen | 60 % (≈ 22‑24 km) | Saturday **15 km** easy + a short 5 km at goal marathon pace. |
| **3** | Light & fast | 45 % (≈ 16‑18 km) | Saturday **12 km** with 2 × 3 km at race pace. |
| **2** | Race‑ready | 30 % (≈ 10‑12 

In [8]:
K.side_by_side("maya-6", with_mem0_2, from_scratch_4)

maya-6, 2026-10-13 -- Maya:
  Final check before I start tapering. Can you remind me: when is my race, what's my goal time, which days do I run
  now and at what time of day, and how have my long runs built up so far? And any advice for race morning?

---- the coach with memory Mem0 ----
**Race & Goal**  
- **Riverside Half Marathon:** **Sunday, Nov 1 2026**  
- **Target finish:** **1 hour 45 minutes** (≈ 5 min /km or 8 min /mile)

**Current weekly schedule** (all **morning runs before work**)  
| Day | Run type | Approx. duration |
|-----|----------|------------------|
| **Tuesday** | Easy/tempo (6‑8 km) | 30‑40 min |
| **Friday**   | Hill repeats or speed work (5‑7 km) | 35‑45 min |
| **Sunday**   | Long run (currently 18 km, tapering soon) | 1 h 45‑2 h |

**Long‑run progression so far**  

| Date | Distance | Time | Notes |
|------|----------|------|-------|
| Aug 30 (Sun) | 14 km | 1:26 | First long run of the block |
| Sep 6 (Sun)  | 16 km | 1:38 | Felt good, no knee issues |
| Se

In [9]:
for runner in ("maya", "tom"):
    held_4 = S.holdings(runner, S.ROUND4)
    print(runner)
    for field, fact in held_4["profile"].items():
        print(f"  {field:16} {fact['value']}   (since {fact['since']}; before: {fact['was']})")
    for run in held_4["runs"]:
        print("  ", run["on_day"], run["kind"], run["km"], "km", run["minutes"] or "", run["felt"])
    for note in held_4["notes"]:
        print("   note:", note["at"], note["text"])
    print()
K.show(K.cost(with_mem0_2, from_scratch_4)["the scratch coach, on Groq"])

maya
  race             Riverside Half Marathon on Sunday 1 November   (since 2026-09-29; before: Riverside Half Marathon on Sunday 1 November)
  goal             1:45   (since 2026-09-29; before: under 1:50)
  days             Tuesday, Friday, Sunday   (since 2026-09-29; before: Tuesday, Friday, Sunday)
  time_of_day      mornings   (since 2026-09-29; before: mornings)
  injuries         knee is fine   (since 2026-09-29; before: knee is fine)
  longest_run      18 km in 1:51   (since 2026-09-29; before: 18 km in 1:51)
  job              started new job   (since 2026-09-29; before: started new job)
  10k_pb           49:30   (since 2026-09-29; before: None)
   2026-08-30 long 14.0 km 86.0 
   2026-09-01 easy 6.0 km  
   2026-09-03 easy 6.0 km  
   2026-09-06 long 16.0 km 98.0 felt good
   2026-09-13 long 17.0 km 105.0 
   2026-09-20 long 18.0 km 111.0 
   2026-09-26 race 10.0 km 49.5 
   note: 2026-10-13 Maya is preparing to taper before the Riverside Half Marathon.
   note: 2026-09-29

## 16 · Measuring every round against an answer key

Reading replies side by side finds problems; it does not count them. So every round is now measured against an **answer key**: the 50 important facts in the runners' messages, written by hand from the messages (`answer_key.py`) and never shown to either system.
- Facts are split where their parts can be wrong on their own: a run's distance, date and time are three facts.
- Each fact carries the exact words it comes from, and code checks those words are in the message.
- States (race date, goal, days) carry the dates each was true, so "current" and "stale" mean something on any given day.

Three measures, kept apart, because a system can save a fact right and still show it wrong later:
- **Extraction.** Per fact, did that check-in's write keep it right, keep it wrong, or not keep it at all? A check-in is marked only on its own message, so a later correction does not mend an earlier write. Facts invented as the runner's are counted separately.
- **Current-state accuracy.** At the two check-ins that ask about the present, did the reply have what was true that day?
- **Stale exposure.** At each check-in after a change, how many items in the coach's context stated a replaced value as if it were current? A change stated as a change, and a value labelled "before:", do not count.

The marks are by hand, each with its reason, in `baseline.py`:

In [2]:
import answer_key as A
import baseline as B

print(A.check(), "facts, each checked against its message")
B.check()
for row in B.table():
    print(row)

50 facts, each checked against its message
{'run': 'mem0 r1', 'saved right': '50/50', 'saved wrong': 0, 'not saved': 0, 'invented': 1, 'current state right': '9/10', 'stale items shown': '16/69 (23%)'}
{'run': 'mem0 r2', 'saved right': '48/50', 'saved wrong': 2, 'not saved': 0, 'invented': 0, 'current state right': '10/10', 'stale items shown': '8/46 (17%)'}
{'run': 'scratch r1', 'saved right': '28/50', 'saved wrong': 7, 'not saved': 15, 'invented': 0, 'current state right': '5/10', 'stale items shown': '5/46 (11%)'}
{'run': 'scratch r2', 'saved right': '38/50', 'saved wrong': 7, 'not saved': 5, 'invented': 1, 'current state right': '8/10', 'stale items shown': '1/45 (2%)'}
{'run': 'scratch r3', 'saved right': '38/50', 'saved wrong': 5, 'not saved': 7, 'invented': 0, 'current state right': '9/10', 'stale items shown': '4/51 (8%)'}
{'run': 'scratch r4', 'saved right': '45/50', 'saved wrong': 1, 'not saved': 4, 'invented': 0, 'current state right': '9/10', 'stale items shown': '2/59 (3%)

The measures put the two designs' weaknesses in different places:
- **Mem0's weakness is not extraction; it is old facts.** It saved 50 and 48 of the 50 facts right, but showed the coach the most superseded facts as if current (17–23% of its context): "signed up for the race on 18 October", "runs four days, in the evening". That is the add-only design, measured. Its coach still answered right, because the corrections came back too.
- **The scratch build's weakness is extraction, and each round improved it:** 28, 38, 38, then 45 facts saved right. Its stale exposure stays low because the profile replaces.

What round 4 still missed was a pattern: habits and small qualifiers. "As I always seem to at race starts" was never saved, Tom's cramps were saved as a one-off instead of a habit, and "hills included", "long run on Sunday" and "first marathon" were dropped.

## 17 · Round 5: what goes where, habits, and notes that go stale

Round 5 targets what the measures found. Everything general went into chapter 4:

- **In `writing.py`:**
  - `rules`: what belongs in each store. The profile holds current states, the events log holds what happened, and notes hold lasting facts that are neither, one each, never restating a field.
  - `Note`: a habit is a note of its own kind.
  - `Statement` and `uncovered`: every statement in the message is listed with where it went, and code checks each one is really in the form. A missing one is sent back.
  - `derived_problems`: a new field the records could work out (longest run, totals, best times) is refused.
- **In `stores.py`:**
  - `QdrantJournal.retire`: a note names the profile fields it depends on, and is taken out of every search when one of them changes.
  - `Profile` ignores a value that has not changed, and keeps every earlier one.

What is the coach's own: the running fields and examples, runs in the runner's words, and running's extra derived words (pb, mileage, weekly, pace).

One more change, found building this round: chapter 4's check now treats a plural as its singular ("Sundays" is "Sunday"). Round 4 ran under the stricter check.

```python
# writing.py
def rules(events, example_states, example_habit):
    """The placement rules for a form with a profile, an events log and notes. `events` names the log ("runs"), the
    examples are the domain's own.

    The events line says THIS message's events, not merely the ones that happened: asked for "every one that
    happened", a model shown the record copied every earlier run back into the form, at six of nine check-ins in
    `06_mem0`'s fifth round -- each refused by the checks, each costing a second call."""
    return f"""Each thing goes in ONE place:
- profile: the current state of something that stays true until it changes ({example_states}). Answer EVERY field:
  "unchanged", or its whole new value, complete on its own. A day, date, number or name in a value must be one the
  message or the record says.
- new_fields: a current state that fits none of the fields, with its name, what it holds and its value. Never
  something the {events} can work out (longest, totals, averages, best, latest).
- {events}: every one THIS message reports -- never the ones already on record.
- notes: something lasting that is neither a state nor one of the {events} -- ONE fact per note. A habit (something
  that keeps happening, e.g. "{example_habit}") is a note of kind "habit". A note never restates a profile field.
  In `depends_on`, list the profile fields the note would stop being true without.
- statements: every statement made about themselves, each in a few words, with where it went: "profile" (and its
  field), "events" (the {events}), "note", or "nothing" (a question, or not worth keeping)."""

def uncovered(form, fields, events, on_record):
    """Statements the form says went somewhere they did not: a profile field neither changed nor already holding
    something, an events log with nothing in it, notes with none. `on_record(field)` says whether a field already
    holds a value -- a statement repeating what is known can go to an unchanged field."""
    done = {f for f, _ in changed(form, fields)}
    return [{"key": ("statement", s.said), "what": f"'{s.said}' -> {s.went_to} {s.field}".strip(),
             "why": "it is not in the form"}
            for s in form.statements
            if (s.went_to == "profile" and key(s.field) not in done and not on_record(key(s.field)))
            or (s.went_to == "events" and not events) or (s.went_to == "note" and not form.notes)]
```

```python
# coach_scratch.py
def card5_step(memory, session):
    """Round 5's writing: round 4's form with the store rules, habits, notes that depend on fields, and every
    statement accounted for -- checked in code and sent back once if anything fails."""
    from datetime import date

    import writing as W
    name, message, today = R.RUNNERS[session["runner"]], session["message"], session["date"]
    fields = memory.profile.fields()
    profile, runs = memory.lines()
    record = "\n".join(["Profile:", *(profile or ["- nothing yet"]), "Runs:", *(runs or ["- none logged"])])
    source = message + "\n" + record
    form = W.profile_form(fields, runs=(list[SaidRun], Field(default_factory=list)),
                          notes=(list[W.Note], Field(default_factory=list)),
                          statements=(list[W.Statement], Field(default_factory=list)))

    def problems(card):
        found = W.profile_problems(card, fields, source) + W.derived_problems(card, RUNNING_DERIVED)
        for i, run in enumerate(card.runs):
            # a run whose distance today's message never gives was not reported today -- copied from the record --
            # so it is dropped, not sent back: asking again would cost a call to remove it
            if f"{run.km:g}" not in W.words(message):
                found.append({"key": ("run", i), "what": f"run '{run.when}' {run.km:g} km",
                              "why": "not in this message", "drop": True})
            elif W.day_of(run.when, today) is None:
                found.append({"key": ("run", i), "what": f"run '{run.when}'", "why": "no day can be worked out from these words"})
        return found + W.note_problems(card, fields, source) + W.uncovered(card, fields, card.runs, memory.profile.read)

    prompt = CARD5.format(name=name, today=f"{date.fromisoformat(today):%A} {today}", record=record, message=message)
    card, still, sent_back = W.fill(prompt, form, problems)
    refused = {p["key"] for p in still}
    for new in card.new_fields:
        if W.key(new.name) and W.key(new.name) not in fields and ("profile", W.key(new.name)) not in refused:
            memory.profile.add_field(W.key(new.name), new.about)
    changes = [(f, v) for f, v in W.changed(card, fields) if ("profile", f) not in refused]
    kept_runs = [Run(on=W.day_of(r.when, today), kind=r.kind, km=r.km, minutes=minutes_of(r.time), felt=r.felt)
                 for i, r in enumerate(card.runs) if ("run", i) not in refused]
    kept_notes = [n for i, n in enumerate(card.notes) if ("note", i) not in refused]
    said = memory.keep5(today, changes, kept_runs, kept_notes)
    return {"card": card.model_dump(), "sent back": sent_back,
            "refused": [{k: v for k, v in p.items() if k != "key"} for p in still], "said": said}
```

In [3]:
from_scratch_5, finished = S.run(root=S.ROUND5, path=S.ANSWERS5, write="card5")
print("\nall nine done" if finished else "\nstopped -- run this cell again to carry on")

  maya-1: kept 1 time(s), 2 calls, 4,816 tokens


  tom-1: kept 1 time(s), 2 calls, 4,078 tokens


  maya-2: kept 1 time(s), 2 calls, 4,238 tokens


  maya-3: kept 1 time(s), 3 calls, 9,397 tokens


  tom-2: kept 1 time(s), 3 calls, 7,981 tokens


  maya-4: kept 1 time(s), 3 calls, 9,443 tokens


  maya-5: kept 1 time(s), 3 calls, 9,194 tokens


  tom-3: kept 1 time(s), 3 calls, 6,379 tokens


  maya-6: kept 1 time(s), 3 calls, 9,292 tokens

all nine done


In [3]:
for sid, record in from_scratch_5.items():
    step = record["kept"][0]
    for line in step["sent back"]:
        print(sid, "sent back:", line)
    for refused in step["refused"]:
        print(sid, "REFUSED:  ", refused)

maya-3 sent back: run '2026-08-30' -- no day can be worked out from these words
maya-3 sent back: run '2026-09-01' -- no day can be worked out from these words
maya-3 sent back: run '2026-09-03' -- no day can be worked out from these words
maya-3 sent back: run '2026-09-06' -- no day can be worked out from these words
maya-3 sent back: run '2026-09-13' -- no day can be worked out from these words
maya-3 REFUSED:   {'what': "run 'Sunday' 14 km", 'why': 'the distance is not in the message'}
maya-3 REFUSED:   {'what': "run 'Tuesday' 6 km", 'why': 'the distance is not in the message'}
maya-3 REFUSED:   {'what': "run 'Thursday' 6 km", 'why': 'the distance is not in the message'}
maya-3 REFUSED:   {'what': "run 'Sunday' 16 km", 'why': 'the distance is not in the message'}
tom-2 sent back: run '2026-08-29' -- no day can be worked out from these words
maya-4 sent back: run '2026-08-30' -- no day can be worked out from these words
maya-4 sent back: run '2026-09-01' -- no day can be worked out f

In [4]:
K.side_by_side("maya-2", with_mem0_2, from_scratch_5)

maya-2, 2026-09-08 -- Maya:
  Sunday's long run was 16 km in 1:38 -- felt good, and the knee was fine. Tuesday and Thursday were easy 6 km runs.
  Should I add a tempo run, and if so, on which day?

---- the coach with memory Mem0 ----
Great job on the 16 km long run—16 km in 1:38 with a clean knee is a solid step toward that sub‑1:50 half‑marathon goal. Your easy 6 km runs on Tue/Thu look good for recovery, too.

**Tempo suggestion**

| Day | Session | How to fit it in |
|-----|---------|------------------|
| **Thursday** | **Tempo** (after a 1‑km easy jog) | 5 km at half‑marathon goal pace ≈ 5:10–5:15 / km, then 1 km cool‑down. Keep the total Thursday mileage around 7 km so you stay easy overall. |
| **Saturday** | Keep as easy/stride day | 6 km easy + 4–6 × 100 m strides to keep leg turnover sharp without extra fatigue before Sunday’s long. |

**Why Thursday?**  
- It gives you a solid mid‑week quality effort without cramming it right before the long run.  
- You still have a recove

In [5]:
K.side_by_side("maya-4", with_mem0_2, from_scratch_5)

maya-4, 2026-09-22 -- Maya:
  Good news: I saw a physio and my knee is fine -- it was a tight IT band, and I've been cleared to run normally,
  hills included. Less good: I've started a new job, so I can only run three days a week now -- Tuesday, Friday and
  Sunday. Sunday's long run was 18 km in 1:51. Can you rework my week around the new days?

---- the coach with memory Mem0 ----
Hey Maya – great news about the IT‑band and getting cleared for hills! With the new three‑day schedule we can still hit the quality sessions you need for a sub‑1:50 half‑marathon while keeping the load manageable for your knee.

**This week (Tue / Fri / Sun)**  

| Day | Workout | Why it helps | Approx. km |
|-----|----------|--------------|------------|
| **Tuesday** | **Easy + Strides** – 8 km total: 6 km easy (≈6:30‑6:45 / km) + 4 × 100 m strides (slow jog back) | Keeps aerobic base, promotes good form, flushes out any residual tightness | 8 |
| **Friday** | **Hill Repeats** – 1 km warm‑up, 6 × 90 s uph

In [6]:
K.side_by_side("maya-5", with_mem0_2, from_scratch_5)

maya-5, 2026-09-29 -- Maya:
  I ran a 10k race on Saturday in 49:30 -- a personal best! I went out far too fast in the first kilometre, as I
  always seem to at race starts, and paid for it at the end. Because of that result I've decided to go for 1:45 at
  the half instead of 1:50. Is that realistic, and how should my next few weeks look?

---- the coach with memory Mem0 ----
**Congrats on the 10 K PB, Maya!** A 49:30 finish (≈4:57 km⁻¹) shows you’re already right on the pace you need for a 1:45 half (≈4:58 km⁻¹). The big question is turning that speed into sustained endurance while keeping the IT‑band happy.

---

### Is 1:45 realistic?
- **Current fitness:** 10 K at 4:57 km⁻¹ → you have the speed.
- **Long‑run pace:** 18 km in 1:51 = 5:12 km⁻¹, about 15 s slower than goal.
- **Gap to close:** ≈15 s/km over the second half of a race. With a few targeted workouts and a proper taper, a 1:45 is **within reach**.

---

## 3‑day‑a‑week plan (Tue / Fri / Sun) – next 5 weeks

| Week | Tue –

In [7]:
K.side_by_side("tom-3", with_mem0_2, from_scratch_5)

tom-3, 2026-09-30 -- Tom:
  Quick one: what's my longest run been so far, and how many weeks until my race? I want to plan my taper. And what
  should I eat on race morning, and when?

---- the coach with memory Mem0 ----
**Longest training run so far:**  
- **28 km** on Saturday 12 Sept (the run where you switched to energy gels).

**Time to the City Marathon (Nov 8, 2026):**  
- From today (30 Sept) it’s **5 weeks + 2 days** – essentially a 5‑week taper window.

---

### Quick taper outline (5 weeks → race)

| Week | Goal | Mileage (≈ % of peak week) | Key workouts |
|------|------|----------------------------|--------------|
| **5** (now) | Begin dropping volume | 80 % of peak (≈ 30‑32 km total) | Keep the Saturday long run at **20 km** (easy pace). |
| **4** | Sharpen | 60 % (≈ 22‑24 km) | Saturday **15 km** easy + a short 5 km at goal marathon pace. |
| **3** | Light & fast | 45 % (≈ 16‑18 km) | Saturday **12 km** with 2 × 3 km at race pace. |
| **2** | Race‑ready | 30 % (≈ 10‑12 

In [8]:
K.side_by_side("maya-6", with_mem0_2, from_scratch_5)

maya-6, 2026-10-13 -- Maya:
  Final check before I start tapering. Can you remind me: when is my race, what's my goal time, which days do I run
  now and at what time of day, and how have my long runs built up so far? And any advice for race morning?

---- the coach with memory Mem0 ----
**Race & Goal**  
- **Riverside Half Marathon:** **Sunday, Nov 1 2026**  
- **Target finish:** **1 hour 45 minutes** (≈ 5 min /km or 8 min /mile)

**Current weekly schedule** (all **morning runs before work**)  
| Day | Run type | Approx. duration |
|-----|----------|------------------|
| **Tuesday** | Easy/tempo (6‑8 km) | 30‑40 min |
| **Friday**   | Hill repeats or speed work (5‑7 km) | 35‑45 min |
| **Sunday**   | Long run (currently 18 km, tapering soon) | 1 h 45‑2 h |

**Long‑run progression so far**  

| Date | Distance | Time | Notes |
|------|----------|------|-------|
| Aug 30 (Sun) | 14 km | 1:26 | First long run of the block |
| Sep 6 (Sun)  | 16 km | 1:38 | Felt good, no knee issues |
| Se

In [9]:
for runner in ("maya", "tom"):
    held_5 = S.holdings(runner, S.ROUND5)
    print(runner)
    for field, fact in held_5["profile"].items():
        print(f"  {field:16} {fact['value']}   (since {fact['since']}; before: {fact['was']})")
    for run in held_5["runs"]:
        print("  ", run["on_day"], run["kind"], run["km"], "km", run["minutes"] or "", run["felt"])
    for note in held_5["notes"]:
        print("   note:", note["at"], f"[{note['source']}]", note["text"], "(retired)" if note.get("retired") else "")
    print()
K.show(K.cost(with_mem0_2, from_scratch_5)["the scratch coach, on Groq"])

maya
  race             Riverside Half Marathon on Sunday 1 November   (since 2026-09-15; before: Riverside Half Marathon on Sunday 18 October)
  goal             finish under 1:45   (since 2026-09-29; before: finish under 1:50)
  days             Tuesday, Friday, Sunday   (since 2026-09-22; before: Tuesday, Thursday, Saturday, Sunday)
  time_of_day      mornings   (since 2026-09-15; before: evening)
  injuries         knee fine   (since 2026-09-22; before: left knee gets sore after hill repeats on Thursday)
   2026-08-30 long 14.0 km 86.0 
   2026-09-01 easy 6.0 km  
   2026-09-03 easy 6.0 km  
   2026-09-06 long 16.0 km 98.0 felt good, knee fine
   2026-09-13 long 17.0 km 105.0 
   2026-09-20 long 18.0 km 111.0 
   2026-09-26 race 10.0 km 49.5 
   note: 2026-09-29 [habit] starts races too fast 
   note: 2026-09-22 [fact] physio cleared to run hills 
   note: 2026-09-15 [habit] knee gets sore after hill repeats on Thursdays (retired)
   note: 2026-09-08 [fact] knee was fine after Sund

## 18 · Round 6: copied runs cost nothing

Round 5's fixes each worked on what they targeted, and overall it scored no better than round 4, at 65k Groq tokens against 40k. The extra cost had one cause. At six of nine check-ins the model copied Maya's earlier runs from the record into the form's "runs to add". The checks refused every copy, so nothing was duplicated, but any refusal sent the whole form back for a second call.

Two changes, both in chapter 4's `writing.py`:
- **The rule's wording.** The events log takes only what *this message* reports, never what is already on record. It used to say every one "that happened".
- **A guard in code, in case the model copies anyway.** A problem marked "drop" is removed without sending the form back. Here, a run whose distance today's message never gives was not reported today, so it is dropped.

Nothing else changes from round 5.

In [2]:
from_scratch_6, finished = S.run(root=S.ROUND6, path=S.ANSWERS6, write="card5")
print("\nall nine done" if finished else "\nstopped -- run this cell again to carry on")
for sid, record in from_scratch_6.items():
    step = record["kept"][0]
    for line in step["sent back"]:
        print(sid, "sent back:", line)
    for refused in step["refused"]:
        print(sid, "dropped or refused:", refused)

  maya-1: kept 1 time(s), 2 calls, 4,195 tokens


  tom-1: kept 1 time(s), 3 calls, 6,955 tokens


  maya-2: kept 1 time(s), 2 calls, 4,140 tokens


  maya-3: kept 1 time(s), 2 calls, 4,791 tokens


  tom-2: kept 1 time(s), 2 calls, 4,318 tokens


  maya-4: kept 1 time(s), 2 calls, 4,533 tokens


  maya-5: kept 1 time(s), 2 calls, 4,798 tokens


  tom-3: kept 1 time(s), 2 calls, 3,746 tokens


  maya-6: kept 1 time(s), 2 calls, 3,875 tokens

all nine done
tom-1 sent back: race = City Marathon on 2026-11-08 -- not said: 2026, 11, 08
tom-1 sent back: days = 5 days a week -- not said: 5
tom-1 dropped or refused: {'what': 'race = City Marathon on 2026-11-08', 'why': 'not said: 2026, 11, 08'}
tom-1 dropped or refused: {'what': 'days = 5 days a week', 'why': 'not said: 5'}


In [10]:
K.side_by_side("tom-3", with_mem0_2, from_scratch_6)

tom-3, 2026-09-30 -- Tom:
  Quick one: what's my longest run been so far, and how many weeks until my race? I want to plan my taper. And what
  should I eat on race morning, and when?

---- the coach with memory Mem0 ----
**Longest training run so far:**  
- **28 km** on Saturday 12 Sept (the run where you switched to energy gels).

**Time to the City Marathon (Nov 8, 2026):**  
- From today (30 Sept) it’s **5 weeks + 2 days** – essentially a 5‑week taper window.

---

### Quick taper outline (5 weeks → race)

| Week | Goal | Mileage (≈ % of peak week) | Key workouts |
|------|------|----------------------------|--------------|
| **5** (now) | Begin dropping volume | 80 % of peak (≈ 30‑32 km total) | Keep the Saturday long run at **20 km** (easy pace). |
| **4** | Sharpen | 60 % (≈ 22‑24 km) | Saturday **15 km** easy + a short 5 km at goal marathon pace. |
| **3** | Light & fast | 45 % (≈ 16‑18 km) | Saturday **12 km** with 2 × 3 km at race pace. |
| **2** | Race‑ready | 30 % (≈ 10‑12 

In [11]:
K.side_by_side("maya-6", with_mem0_2, from_scratch_6)

maya-6, 2026-10-13 -- Maya:
  Final check before I start tapering. Can you remind me: when is my race, what's my goal time, which days do I run
  now and at what time of day, and how have my long runs built up so far? And any advice for race morning?

---- the coach with memory Mem0 ----
**Race & Goal**  
- **Riverside Half Marathon:** **Sunday, Nov 1 2026**  
- **Target finish:** **1 hour 45 minutes** (≈ 5 min /km or 8 min /mile)

**Current weekly schedule** (all **morning runs before work**)  
| Day | Run type | Approx. duration |
|-----|----------|------------------|
| **Tuesday** | Easy/tempo (6‑8 km) | 30‑40 min |
| **Friday**   | Hill repeats or speed work (5‑7 km) | 35‑45 min |
| **Sunday**   | Long run (currently 18 km, tapering soon) | 1 h 45‑2 h |

**Long‑run progression so far**  

| Date | Distance | Time | Notes |
|------|----------|------|-------|
| Aug 30 (Sun) | 14 km | 1:26 | First long run of the block |
| Sep 6 (Sun)  | 16 km | 1:38 | Felt good, no knee issues |
| Se

## 19 · Round 7: a check that compares meaning

Round 6 brought the cost back to 41k tokens, and nothing was copied. But the check threw away correct facts at Tom's first check-in, because it compared how values were *written*:
- he said "8 November", and the model wrote "2026-11-08";
- he said "five days a week", and the model wrote "5 days a week".

Both were refused, sent back and refused again. So his race was never stored, and at his last check-in the coach made up a date: "13 October, two weeks away".

The fix is in chapter 4's `unsupported`: days, dates and numbers are compared by what they mean. "5" matches "five", and "2026-11-08", "8 November 2026" and "Nov 8" are the same day. Number words in a value are still not checked, as before. Nothing else changes from round 6.

In [2]:
from_scratch_7, finished = S.run(root=S.ROUND7, path=S.ANSWERS7, write="card5")
print("\nall nine done" if finished else "\nstopped -- run this cell again to carry on")
for sid, record in from_scratch_7.items():
    step = record["kept"][0]
    for line in step["sent back"]:
        print(sid, "sent back:", line)
    for refused in step["refused"]:
        print(sid, "dropped or refused:", refused)

  maya-1: kept 1 time(s), 2 calls, 4,367 tokens


  tom-1: kept 1 time(s), 2 calls, 4,648 tokens


  maya-2: kept 1 time(s), 2 calls, 4,173 tokens


  maya-3: kept 1 time(s), 2 calls, 4,834 tokens


  tom-2: kept 1 time(s), 3 calls, 7,959 tokens


  maya-4: kept 1 time(s), 2 calls, 5,220 tokens


  maya-5: kept 1 time(s), 2 calls, 4,966 tokens


  tom-3: kept 1 time(s), 2 calls, 3,866 tokens


  maya-6: kept 1 time(s), 2 calls, 3,846 tokens

all nine done
tom-2 sent back: pre_run_meal_timing = 1 hour -- not said: 1


In [12]:
K.side_by_side("tom-3", with_mem0_2, from_scratch_7)

tom-3, 2026-09-30 -- Tom:
  Quick one: what's my longest run been so far, and how many weeks until my race? I want to plan my taper. And what
  should I eat on race morning, and when?

---- the coach with memory Mem0 ----
**Longest training run so far:**  
- **28 km** on Saturday 12 Sept (the run where you switched to energy gels).

**Time to the City Marathon (Nov 8, 2026):**  
- From today (30 Sept) it’s **5 weeks + 2 days** – essentially a 5‑week taper window.

---

### Quick taper outline (5 weeks → race)

| Week | Goal | Mileage (≈ % of peak week) | Key workouts |
|------|------|----------------------------|--------------|
| **5** (now) | Begin dropping volume | 80 % of peak (≈ 30‑32 km total) | Keep the Saturday long run at **20 km** (easy pace). |
| **4** | Sharpen | 60 % (≈ 22‑24 km) | Saturday **15 km** easy + a short 5 km at goal marathon pace. |
| **3** | Light & fast | 45 % (≈ 16‑18 km) | Saturday **12 km** with 2 × 3 km at race pace. |
| **2** | Race‑ready | 30 % (≈ 10‑12 

In [13]:
K.side_by_side("maya-6", with_mem0_2, from_scratch_7)

maya-6, 2026-10-13 -- Maya:
  Final check before I start tapering. Can you remind me: when is my race, what's my goal time, which days do I run
  now and at what time of day, and how have my long runs built up so far? And any advice for race morning?

---- the coach with memory Mem0 ----
**Race & Goal**  
- **Riverside Half Marathon:** **Sunday, Nov 1 2026**  
- **Target finish:** **1 hour 45 minutes** (≈ 5 min /km or 8 min /mile)

**Current weekly schedule** (all **morning runs before work**)  
| Day | Run type | Approx. duration |
|-----|----------|------------------|
| **Tuesday** | Easy/tempo (6‑8 km) | 30‑40 min |
| **Friday**   | Hill repeats or speed work (5‑7 km) | 35‑45 min |
| **Sunday**   | Long run (currently 18 km, tapering soon) | 1 h 45‑2 h |

**Long‑run progression so far**  

| Date | Distance | Time | Notes |
|------|----------|------|-------|
| Aug 30 (Sun) | 14 km | 1:26 | First long run of the block |
| Sep 6 (Sun)  | 16 km | 1:38 | Felt good, no knee issues |
| Se

Every round, measured against the answer key:

In [14]:
import importlib

importlib.reload(B)
B.check()
for row in B.table():
    print(row)
print({run: f"{tokens:,}" for run, tokens in B.TOKENS.items()})

{'run': 'mem0 r1', 'saved right': '50/50', 'saved wrong': 0, 'not saved': 0, 'invented': 1, 'current state right': '9/10', 'stale items shown': '16/69 (23%)'}
{'run': 'mem0 r2', 'saved right': '48/50', 'saved wrong': 2, 'not saved': 0, 'invented': 0, 'current state right': '10/10', 'stale items shown': '8/46 (17%)'}
{'run': 'scratch r1', 'saved right': '28/50', 'saved wrong': 7, 'not saved': 15, 'invented': 0, 'current state right': '5/10', 'stale items shown': '5/46 (11%)'}
{'run': 'scratch r2', 'saved right': '38/50', 'saved wrong': 7, 'not saved': 5, 'invented': 1, 'current state right': '8/10', 'stale items shown': '1/45 (2%)'}
{'run': 'scratch r3', 'saved right': '38/50', 'saved wrong': 5, 'not saved': 7, 'invented': 0, 'current state right': '9/10', 'stale items shown': '4/51 (8%)'}
{'run': 'scratch r4', 'saved right': '45/50', 'saved wrong': 1, 'not saved': 4, 'invented': 0, 'current state right': '9/10', 'stale items shown': '2/59 (3%)'}
{'run': 'scratch r5', 'saved right': '44

## 20 · Retrieval: did what was needed reach the coach?

Extraction says what was saved; the replies say what the coach did with it. In between is retrieval: what the coach was actually shown. At the five check-ins that need memory, `answer_key.NEEDED` lists the 29 facts from earlier check-ins that a good reply needs. Code checks each one was said earlier, and that each state is the one true on the day.

Each was looked for in the coach's context, in its current version. Each miss is labelled by its cause:
- **never saved right**: a write failure, so retrieval had nothing to find;
- **saved, not retrieved**: a retrieval failure.

Code cross-checks the labels against the extraction marks.

In [3]:
B.check_retrieval()
for row in B.retrieval_table():
    print(row)

{'run': 'mem0 r1', 'reached the coach': '29/29', 'never saved (write)': 0, 'saved, not retrieved (retrieval)': 0}
{'run': 'mem0 r2', 'reached the coach': '28/29', 'never saved (write)': 1, 'saved, not retrieved (retrieval)': 0}
{'run': 'scratch r1', 'reached the coach': '17/29', 'never saved (write)': 11, 'saved, not retrieved (retrieval)': 1}
{'run': 'scratch r2', 'reached the coach': '24/29', 'never saved (write)': 5, 'saved, not retrieved (retrieval)': 0}
{'run': 'scratch r3', 'reached the coach': '25/29', 'never saved (write)': 3, 'saved, not retrieved (retrieval)': 1}
{'run': 'scratch r4', 'reached the coach': '26/29', 'never saved (write)': 3, 'saved, not retrieved (retrieval)': 0}
{'run': 'scratch r5', 'reached the coach': '28/29', 'never saved (write)': 1, 'saved, not retrieved (retrieval)': 0}
{'run': 'scratch r6', 'reached the coach': '26/29', 'never saved (write)': 3, 'saved, not retrieved (retrieval)': 0}
{'run': 'scratch r7', 'reached the coach': '28/29', 'never saved (wri

Nearly every miss was a write failure. Only two were retrieval failures, both in the scratch build's notes. But retrieval was hardly tested at this size:
- **The scratch coach is shown its whole profile and run log.** That is injection, not search, so only its notes can be missed.
- **Mem0 holds 8 to 27 memories per runner and returns up to 20**, so its search returns nearly everything.

## 21 · Retrieval with a year behind it

`stress.py` copies each build's best store (Mem0 round 2, scratch round 7) and adds about 100 older memories per runner: a year of runs, races, old goals and schedules, old niggles and preferences, all dated before the scenario.
- Mem0 gets them through `add(infer=False)`, with their names linked as Mem0 links them.
- The scratch build gets the same texts, as runs and notes.

Then Maya's and Tom's last check-ins search again: the two whose stores are complete as of their date. No model is called; it costs about 300 Gemini embeddings, so the cell below reads the saved results. `X.run()` runs it all again.

In [4]:
import json

import stress as X

stressed = json.loads(X.RESULTS.read_text(encoding="utf-8"))     # X.run() to measure again (~300 embeddings)
print("older memories added per runner:", stressed["memories per runner"])
for build in ("mem0", "scratch", "scratch, notes by share", "scratch, hybrid per question + habits"):
    for sid in X.CHECKINS:
        needed = stressed[build][sid]["needed"]
        print(f"{build:40} {sid:7} {sum(needed.values())} of {len(needed)}   missing: {[k for k, v in needed.items() if not v]}")

older memories added per runner: {'maya': 99, 'tom': 100}
mem0                                     maya-6  5 of 9   missing: ['runs Tue, Fri, Sun', 'runs in the mornings', '17 km on 13 September', '18 km on 20 September']
mem0                                     tom-3   4 of 5   missing: ['vegetarian']
scratch                                  maya-6  8 of 9   missing: ['starts races too fast']
scratch                                  tom-3   4 of 5   missing: ['cramps when he eats under 2 hours before']
scratch, notes by share                  maya-6  8 of 9   missing: ['starts races too fast']
scratch, notes by share                  tom-3   4 of 5   missing: ['cramps when he eats under 2 hours before']
scratch, hybrid per question + habits    maya-6  9 of 9   missing: []
scratch, hybrid per question + habits    tom-3   5 of 5   missing: []


Every context was also read, not only matched. The one correction is the race-start habit for Mem0: its "too fast" is the one-off in the 10k memory, a write failure, as in section 20. So Mem0 has 4 of Maya's 9, not 5.

- **Mem0's one search lost current facts and kept old ones.** 13 and 14 of its 20 results were old memories. Maya's current days, her mornings, and her 17 and 18 km runs fell out, and so did Tom's "vegetarian". Meanwhile "runs Tuesday, Thursday, Saturday and Sunday, in the evening" and "signed up for 18 October, under 1:50" stayed, with nothing left beside them to correct them.
- **The scratch build's profile and run log were untouched.** They are shown, not searched: chapter 4's "what will be asked decides the store" pays off here. Its notes lost both habits, though. They ranked 13th and 10th by meaning, below old notes like "ran even splits at the Lakeside 10k".
- **A larger share of notes did not help.** Thirty percent of the notes, at most five, still stopped above the 10th place.
- **Changing the ranking did help.** Chapter 4's `Recall` is now hybrid by default, and searches one question at a time. Asked alone, "any advice for race morning?" ranks Maya's habit first. The coach also shows every habit note, searched or not, because a runner has few habits and they bear on every check-in. With these, every needed fact reached the coach: 14 of 14.

What the new search shows for Maya's last check-in:

In [5]:
print(stressed["scratch, hybrid per question + habits"]["maya-6"]["context"].split("Habits:")[1])


- 2026-09-29 always starts races too fast
Notes that bear on this message:
- 2026-04-14 Maya likes a podcast on her easy runs and music on hard ones (noted at a check-in)
- 2025-11-27 Maya ran even splits at the Lakeside 10k and finished strongly, she told the coach
- 2026-02-04 Maya uses a running watch to keep her easy runs easy
- 2025-10-27 Maya struggles to run well in heat above 25 degrees


---
## Findings

Every reply below was read in full. Each round ran once, so each result is one sample: a fact or two either way is within noise. Section 16 has every round's measures; the table below is the five replies that need memory, as read, for rounds 1 to 4.

**The five check-ins that need memory, round by round:**

| | Mem0, round 1 | Mem0, round 2 | From scratch, round 1 | From scratch, round 2 | From scratch, round 3 | From scratch, round 4 |
|---|---|---|---|---|---|---|
| **Maya, 13 Oct:** race, goal, days, mornings | all right | all right | race **18 October ✗**, days **Tue/Thu/Sat/Sun ✗**; goal and mornings right | all right | all right | all right |
| **Maya, 13 Oct:** how her long runs built up | 16, 17, 18 km, then a **20 km she never ran ✗**; 14 km left out | 14, 16, 17, 18 km, dates right | 14, 16, 17 km, the 18 km only from a note; dates wrong | every run listed as a "long run", the easy runs and the 10k included; dates wrong | 14, 16, 17, 18 km; dates a day or two off | 14, 16, 17, 18 km, dates right |
| **Maya, 13 Oct:** her habit of starting too fast | named | a gentle first 3 km, habit not named | not used | "start conservatively", habit not named | an easy first 3 km, habit not named | "start a little conservatively"; the habit was never written down |
| **Maya, 22 and 29 Sep:** her week rebuilt around Tue/Fri/Sun | right | right | a thin reply after the loop was stopped; then a **four-day plan ✗**, the new days only in a footnote | right | right | right |
| **Tom, 30 Sep:** longest run, race date, cramps | 28 km, 8 November, cramps | the same | 28 km; race **"about 2 weeks away" ✗**; cramps | 28 km, 8 November, cramps; but **"you've done well with water only" ✗** | 28 km, 39 days to go, cramps, gels, vegetarian | 28 km, 5 weeks 4 days, gels, vegetarian; cramps only as "keep the stomach happy" |
| **Tom never told anything of Maya's** | yes | yes | yes | yes | yes | yes |

The coach model also slipped where memory was not involved: both coaches miscounted the weeks to Tom's race in rounds 1 and 2 (5 weeks 2 or 3 days; it is 5 weeks 4 days), one wrote 28 km "on 2024-09-12", and round 3's last reply to Maya turned a 1:45 half into "4:05 per km" (it is about 5:00). Memory gave it the right goal; the arithmetic was its own.

**What decided it was the writing, not the store.** In round 1 neither store lost or mixed up anything it was given; what differed was what went in. Mem0 gives writing a step of its own: after every check-in, a separate model whose only job is memory. It caught every fact, including Tom's cramps, mentioned in passing in a message about something else. The scratch coach wrote memory as a side job of replying, and wrote it carelessly:
- the race saved without its date;
- the race move never written;
- changed days put in a note instead of the profile;
- runs dated wrong.

The profile then showed the coach the old facts as current, and the coach repeated them. Round 2 gave the scratch build a write step of its own (one `extract()` after the reply), and every named fact came out right.

**Mem0 only adds, and the coach model works out what is current.** In 2.2.1, `add` never updates or deletes, so Maya's memory holds both race dates, both goals and both sets of days. Both Mem0 replies to her last check-in still gave the current ones, for two reasons: Mem0's extraction writes a change as a change ("changed from 18 October to Sunday 1 November"), and every memory came with the date of its check-in. The cost is that everything comes back, every time. The 0.1 threshold filtered nothing: at every check-in, the least similar memory returned still scored between 0.51 and 0.60. By Maya's fifth check-in, search was returning its limit of 20, out of the 22 memories she had. At a larger scale, that prompt is where old and new facts compete. The scratch profile shows the coach only the current value, provided the write was right.

**By default, Mem0 also keeps the coach's own advice.** Its prompt asks for "plans or schedules created for the user" from the assistant's messages. In round 1 that was 10 of Maya's 26 memories, and the coach twice read a run it had planned as one she had done. One custom instruction removed it all: 15 memories in round 2, all hers, and no invented runs. Any assistant whose replies contain plans should set this.

**A form filled by the coach model is only as good as its filling.** In round 2, the scratch write step still:
- dated runs wrong, even when told the weekday ("last Sunday" from Tuesday 1 September became Saturday 29 August);
- made up detail: Tom's days as Monday to Friday, though his long runs are on Saturdays;
- left a field stale when the change came under another heading: "gels worked" went into a run's note, so `food` still said water only.

A stale profile field is shown to the coach as a current fact, which is worse than Mem0's two versions with dates beside them. Replacing by field is only as good as the write. Mem0's extraction model (Gemini flash-lite, with a 7,600-token prompt) dated every run right in round 1, and all but one pair in round 2.

**Checking each write fixed what it was aimed at, and cost a little.** Round 3 gave every field a slot, checked every change against the runner's own words, and kept the old values and the runner's messages in view. Each of round 2's wrong facts went:
- `food` became "gels" when Tom said gels worked;
- his days stayed "five days a week";
- every changed field showed what it said before, and since when.

Tom's last reply was the best of all six runs: the only one to count 39 days to his race correctly, and it had his cramps, his gels and that he is vegetarian.

The check also refused one correct fact. The model's quote was "I've done the long runs" where Tom wrote "I've done my long runs", so "vegetarian" never reached his profile. His reply still got it right from his own kept message. A check on exact words is strict by design: here it cost one fact, against the invented one it was there to stop.

Keeping every message also crowded `Recall`'s three slots: at Maya's last check-in all three were her own old messages, and her race-start note did not get in. The prompts grew too: 38k Groq tokens against round 2's 27k. Run dates were not part of this round, and are still a day or two off. The repair for those is to have the model give the runner's phrase ("last Sunday") and let code work out the date.

**Round 4 undid round 3's three mistakes, and moved dates into code.** The general parts went into chapter 4: `Profile`, and `writing.py`'s form, check, date reader and one-retry fill. With them:
- every run was dated right, for the first time in any scratch round;
- diet and race fuel stayed apart: "vegetarian" survived Tom's switch to gels;
- nothing correct was thrown away. Two notes were sent back for dates the model had worked out itself, and both came back fixed.

The form also added fields. One was exactly the case they are for: Tom's `stomach_cramps`. Others were not needed: Maya's `longest_run` copies what the run log already holds, and a copy can go stale.

Three things it still got wrong:
- **Repeated values wiped the history.** At Maya's 29 September check-in, the form re-wrote every field with its *unchanged* value. That reset every "since" date, and the one step of history kept now says "before: 1 November", not 18 October. `Profile` should ignore a value that has not changed.
- **Notes can repeat a profile fact and go stale.** A 1 September note, "runs Tuesday, Thursday, Saturday and Sunday evenings", was brought back at her last check-in. The coach trusted the profile, but nothing replaces a note.
- **Maya's race-start habit was never written down at all.**

Round 4 cost 40k Groq tokens in 21 calls.

**Rounds 5 to 7, measured against the answer key (section 16).**
- **Round 5** added chapter 4's store rules, habits as their own kind, a check that every statement is accounted for, notes retired when their field changes, and no fields the run log could work out. Habits were saved for the first time, and an old knee note retired by itself. But at six check-ins the model copied earlier runs back into the form, and every copy cost a second call: 44 of 50 facts saved right, at 65k tokens.
- **Round 6** asked only for this message's runs, and dropped a copied run without a second call: back to 41k tokens. But the check compared how values were *written*. Tom's race written "2026-11-08" was refused against his "8 November", and "5 days a week" against "five". So his race was never stored, and his coach made one up.
- **Round 7** compares what days, dates and numbers *mean*: 47 of 50 saved right, nothing wrong or invented, every current state right, no stale item in the coach's context, at 44k tokens.

**Where the two builds ended, each at its best round:**

| | Mem0, round 2 | From scratch, round 7 |
|---|---|---|
| Facts saved right (of 50) | 48 | 47 |
| Current state right (of 10) | 10 | 10 |
| Stale items in the coach's context | 8 of 46 (17%) | 0 of 44 |
| Model cost per round | Groq 14k + Gemini about 90k | Groq 44k |
| What keeps the coach current | the coach model, reading old and new facts side by side, with dates | the stores: a field holds only its current value, with its history; a note on a changed field is retired |

Both end up right. Mem0 gets there with almost no design, by extracting everything and leaving the coach to sort out which version is current. The scratch build gets there by designing what goes where, and by checking every write in code. It took seven rounds of measuring to find what to check. The facts it still drops are small ones: her personal best, the new job, the long run being on Sundays.

**Retrieval (sections 20 and 21).** In the nine check-ins, nearly every fact the coach missed was never saved right; retrieval missed two, both scratch notes. With a year of older memories added, it split the other way:
- **Mem0's single search** dropped Maya's current days, mornings and two runs, and Tom's "vegetarian", while it kept her old days and race date. So 8 of 14 needed facts reached its coach, down from 13.
- **The scratch build's profile and run log** were unaffected. Its notes lost both habits, until chapter 4's `Recall` became hybrid by default and searched one question at a time, and the coach showed habits always. Then 14 of 14 reached its coach.

What the old memories showed is that only what is searched can be crowded out. Facts with names are safer shown than searched.

**Notes, the facts nobody designs a field for.** Tom's cramps reached his race-morning answer in all four runs. Mem0 extracted them unasked; the scratch build kept them as a note, and `Recall` brought it back. Maya's habit of starting too fast was less safe. In round 1 Mem0 named it, while the scratch build's note never reached her final check-in. In round 2 neither named it: Mem0 kept it only inside the 10k memory, and the scratch note again missed `Recall`.

**Cost.** On Groq, per round:
- the Mem0 coach: 14–16k tokens in 9 calls;
- the scratch coach: 25k (round 1), 27k (round 2), 38k (round 3) and 40k (round 4) in 17–21 calls, a reply plus a write each check-in, and a second try when a write was sent back.

Mem0's own model took about 90k input tokens per round on Gemini, around 10k for every `add`, because the extraction prompt alone is about 7,600 tokens. That is also why it cannot run on Groq's free tier. Embeddings: 155 texts in round 1, including the names spaCy found.

**Setting Mem0 up** (section 1 has each one):
- the extraction prompt is too large for Groq's free tier;
- `timestamp` raises an error in the open-source build;
- telemetry is on by default, and the history database is machine-wide;
- full search needs Qdrant, fastembed and spaCy, and falls back to meaning-only with a single warning line;
- `get_all` stops at 20 without saying so;
- the links between memories that the prompt asks the model for are thrown away.

**What it added to the repo.** Chapter 4's `stores.py` now has `Profile` (named facts that keep their previous value and dates), and a new `writing.py` checks a write before it is kept. `stores.py` also has `QdrantJournal`: the journal on one engine, with `Journal`'s methods, so `Recall` runs on either. Offline the two journals answer the same, except that Qdrant's BM25 stems words, and its own fusion constant is 2 rather than 60.